In [138]:
import polars as pl
import pandas as pd
import numpy as np
import os

DATA_PATH = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\dataset\sales_pers.item_chunk_0.parquet"

df = pl.read_parquet(DATA_PATH)
print(f"Số dòng: {df.height:,}, Số cột: {df.width}")


Số dòng: 27,332, Số cột: 34


In [139]:
print("\nCác cột ban đầu:")
print(df.columns)


Các cột ban đầu:
['p_id', 'item_id', 'price', 'category_l1_id', 'category_l1', 'category_l2_id', 'category_l2', 'category_l3_id', 'category_l3', 'category_id', 'category', 'description', 'brand', 'manufacturer', 'creation_timestamp', 'is_deleted', 'created_date', 'updated_date', 'sync_status_id', 'last_sync_date', 'sync_error_message', 'image_url', 'gender_target', 'age_group', 'item_type', 'gp', 'weight', 'color', 'size', 'origin', 'volume', 'material', 'sale_status', 'description_new']


# Task 1: Loại bỏ các cột mà nhóm nghĩ là không cần thiết.

In [140]:
# --- Danh sách cột cần loại bỏ (theo phân tích EDA & tương quan) ---
cols_to_drop = [
    # Cột phân cấp danh mục (trùng lặp với category)
    "category_l2", "category_l3",
    "category_l1_id", "category_l2_id", "category_l3_id",
    "category_id",

    # Cột định lượng thừa / không dùng
    "gp", "weight",

    # Cột metadata hệ thống
    "manufacturer", "is_deleted", "sync_status_id",
    "sync_error_message", "image_url", "last_sync_date",
    "volume", "creation_timestamp", "updated_date", "created_date",

    # Cột ID nội bộ
    "p_id"
]


# --- Loại bỏ các cột không cần thiết ---
df_cleaned = df.drop(cols_to_drop)

print(f"\nĐã loại bỏ {len(cols_to_drop)} cột không cần thiết.")
print(f"Số cột còn lại: {df_cleaned.width}")
print("\nDanh sách cột sau khi loại bỏ:")
print(df_cleaned.columns)

# --- Xuất dữ liệu sau Task 1 ---
OUTPUT_DIR = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\data\item-chunk"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "item_chunk_task1_cleaned.parquet")

df_cleaned.write_parquet(OUTPUT_PATH)

print(f"\nDữ liệu sau Task 1 đã được lưu tại:\n{OUTPUT_PATH}")


Đã loại bỏ 19 cột không cần thiết.
Số cột còn lại: 15

Danh sách cột sau khi loại bỏ:
['item_id', 'price', 'category_l1', 'category', 'description', 'brand', 'gender_target', 'age_group', 'item_type', 'color', 'size', 'origin', 'material', 'sale_status', 'description_new']

Dữ liệu sau Task 1 đã được lưu tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\data\item-chunk\item_chunk_task1_cleaned.parquet


### **Các cột 'color', 'size', 'origin', 'material' sẽ được drop sau, sau khi đã tận dụng hết thông tin**

# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lí outlier đối với cột price

Đối với biến price, dữ liệu có hiện tượng lệch phải mạnh và chứa nhiều outlier.
Dựa trên phương pháp IQR (Interquartile Range), các giá trị nhỏ hơn Q1 − 1.5×IQR và lớn hơn Q3 + 1.5×IQR được xem là bất thường.

Cụ thể, giá > 424,000 được cắt ngưỡng (winsorized) xuống 424,000,
và giá trị bằng 0 được thay bằng giá trung vị (119,000).
Cách xử lý này giúp giảm ảnh hưởng của các điểm cực trị mà vẫn giữ lại thông tin gốc.

In [142]:
# === Cell: Xử lý Outlier cho cột price (IQR-based capping) ===
import polars as pl

# Bắt đầu từ dataframe hiện tại
df = df_cleaned  

# 1. Tính thống kê cơ bản
price_q1 = df.select(pl.col("price").quantile(0.25)).item()
price_q3 = df.select(pl.col("price").quantile(0.75)).item()
iqr = price_q3 - price_q1

lower_bound = max(0, price_q1 - 1.5 * iqr)
upper_bound = price_q3 + 1.5 * iqr
median_price = df.select(pl.col("price").median()).item()

print("=== THỐNG KÊ GIÁ TRỊ PRICE ===")
print(f"Q1 = {price_q1:,.0f}")
print(f"Q3 = {price_q3:,.0f}")
print(f"IQR = {iqr:,.0f}")
print(f"Ngưỡng dưới: {lower_bound:,.0f}")
print(f"Ngưỡng trên: {upper_bound:,.0f}")
print(f"Giá trung vị: {median_price:,.0f}")

# 2. Áp dụng capping để xử lý outlier
df = df.with_columns([
    pl.when(pl.col("price") <= 0)
      .then(median_price)
      .when(pl.col("price") > upper_bound)
      .then(upper_bound)
      .otherwise(pl.col("price"))
      .alias("price_capped")
])

# 3. In thống kê sau khi xử lý
summary = df.select([
    pl.col("price").mean().alias("mean_before"),
    pl.col("price_capped").mean().alias("mean_after"),
    pl.col("price").max().alias("max_before"),
    pl.col("price_capped").max().alias("max_after"),
])

print("\n=== TÓM TẮT SAU XỬ LÝ ===")
print(summary)

# 4. Cập nhật dataframe chính
df_cleaned = df


=== THỐNG KÊ GIÁ TRỊ PRICE ===
Q1 = 49,000
Q3 = 199,000
IQR = 150,000
Ngưỡng dưới: 0
Ngưỡng trên: 424,000
Giá trung vị: 119,000

=== TÓM TẮT SAU XỬ LÝ ===
shape: (1, 4)
┌───────────────┬──────────────┬───────────────┬───────────┐
│ mean_before   ┆ mean_after   ┆ max_before    ┆ max_after │
│ ---           ┆ ---          ┆ ---           ┆ ---       │
│ f64           ┆ f64          ┆ decimal[38,4] ┆ f64       │
╞═══════════════╪══════════════╪═══════════════╪═══════════╡
│ 190456.829467 ┆ 142756.20379 ┆ 20990000.0000 ┆ 424000.0  │
└───────────────┴──────────────┴───────────────┴───────────┘


In [143]:
# === Cell: Cập nhật giá trị price sau khi xử lý outlier ===
df_final = (
    df_cleaned
    .drop("price")  # bỏ cột gốc
    .rename({"price_capped": "price"})  # thay thế bằng bản đã xử lý
)

print("Đã cập nhật cột 'price' sau khi xử lý outlier.")
print(df_cleaned.select(pl.col("price")).describe())


Đã cập nhật cột 'price' sau khi xử lý outlier.
shape: (9, 2)
┌────────────┬───────────────┐
│ statistic  ┆ price         │
│ ---        ┆ ---           │
│ str        ┆ f64           │
╞════════════╪═══════════════╡
│ count      ┆ 27332.0       │
│ null_count ┆ 0.0           │
│ mean       ┆ 190456.829467 │
│ std        ┆ 511123.971813 │
│ min        ┆ 0.0           │
│ 25%        ┆ 49000.0       │
│ 50%        ┆ 119000.0      │
│ 75%        ┆ 199000.0      │
│ max        ┆ 2.099e7       │
└────────────┴───────────────┘


## Phân tích cơ bản sau khi EDA và tương quan dữ liệu

Trong phần EDA, chúng tôi có thấy rằng description & description_new có độ tương quan khá cao (0.74), và có độ tương quan với các cột: brand, gender_target,age_group, item_type, color, size, origin, material khá cao (từ 0.4 và có thể lên tới 0.78).

Vì vậy, chúng tôi sẽ tiến hành tìm hiểu sâu hơn về 2 cột description và description_new này để xem có thể xử dụng chúng cho một số mục đích khác sau này không.

In [144]:
pl.Config.set_tbl_rows(50)             # số dòng tối đa hiển thị
pl.Config.set_tbl_cols(10)             # số cột tối đa hiển thị
pl.Config.set_tbl_width_chars(200)     # tổng chiều rộng bảng ký tự
pl.Config.set_fmt_str_lengths(2000)     # độ dài chuỗi tối đa hiển thị (quan trọng nhất!)

df_cleaned.select(["description", "description_new"]).head(5)

description,description_new
str,str
"""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cut (Trên 9 tháng) Loại sản phẩm: Núm ty Đối tượng sử dụng: Trẻ trên 9 tháng Thương hiệu: DrBrown's Điểm nổi bật: Núm vú silicon siêu mềm giúp phát triển cơ hàm của bé thông qua sự cử động lưỡi tự nhiên. Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cut (Trên 9 tháng) là núm ty dành cho trẻ trên 9 tháng của thương hiệu DrBrown's đến từ Mỹ. Núm vú silicon siêu mềm giúp phát triển cơ hàm của bé thông qua sự cử động lưỡi một cách tự nhiên. Núm ty silicon mềm mại Núm vú silicon siêu mềm giúp phát triển cơ hàm của bé thông qua sự cử động lưỡi một cách tự nhiên. Thiết kế đầu núm vừa vặn với vòm miệng bé, mang lại sự thoải mái khi sử dụng. Thiết kế thông minh Núm ty DrBrown's Options Plus được thiết kế với lỗ tiết sữa Y cut, phù hợp với nhu cầu ăn cháo đặc của bé trên 9 tháng tuổi, từ đó giúp bé có cảm giác gần gũi hơn trong từng lần bú. Hướng dẫn sử dụng Rửa sạch núm ty bằng nước rửa chuyên dụng trước khi sử dụng lần đầu. Đun sôi núm vú trong vòng 3-5 phút để tiệt trùng sau mỗi lần sử dụng. Kiểm tra núm vú trước khi cho bé sử dụng, đảm bảo không có dấu hiệu rách hoặc thủng. Không cho trẻ bú bình khi không có sự giám sát của người lớn. Hướng dẫn bảo quản Bảo quản ở nhiệt độ phòng, nơi khô ráo, thoáng mát. Tránh ánh nắng mặt trời trực tiếp. Thay núm ty mới sau khi sử dụng từ 1 - 2 tháng để đảm bảo vệ sinh cho trẻ. Lưu ý Không rửa núm ty bằng xà phòng hoặc nước tẩy rửa. Không tiệt trùng núm ty bằng lò vi sóng. Kiểm tra núm vú thường xuyên trước khi sử dụng. Thành phần Silicon cao cấp Không chứa BPA Chất đàn hồi tốt"""
"""Không xác định""","""Không xác định"""
"""- Chất liệu: Sản phẩm được làm bằng chất liệu silicone mềm, dẻo và nước đã được chưng cất đảm bảo an toàn cho bé. - Dành cho các bé trong giai đoạn mọc răng, giúp làm giảm đau, ngứa lợi cho bé. - Bề mặt có các núm mát-xa nhỏ, mềm mịn, tạo cảm giác dễ chịu cho bé. - Có độ đàn hồi tốt, dễ dàng uốn dẻo, bền đẹp không gây ảnh hưởng tới sức khỏe cho bé. - Thiết kế dạng hình các con vật đồ dùng ngộ nghỉnh, đặc biệt có tay cầm, giúp bé dễ cầm nắm. - Trước khi cho bé sử dụng, bạn có thể cho vào tủ lạnh tạo cảm giác mát lạnh bé thích thú Lưu ý: Không sử dụng nước sôi để khử trùng sản phẩm. Xuất xứ: Thái Lan﻿﻿""","""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) Chất liệu: Silicone mềm, dẻo và nước đã được chưng cất Đối tượng sử dụng: Dành cho các bé trong giai đoạn mọc răng Thương hiệu: Papa Xuất xứ: Thái Lan Giá: 45.000₫ **Miếng Gặm Nướu Papa (CEQ004) (Cá hồng)** là một sản phẩm giúp bé giảm đau trong giai đoạn mọc răng của thương hiệu **Papa đến từ Thái Lan**. Sản phẩm được thiết kế đặc biệt để tạo cảm giác dễ chịu cho bé khi nhai. Chất liệu an toàn: Silicone mềm, dẻo Miếng gặm nướu được làm bằng chất liệu silicone mềm, dẻo và nước đã được chưng cất, đảm bảo an toàn tuyệt đối cho sức khỏe của bé. Cảm giác dễ chịu: Bề mặt núm mát-xa Bề mặt có các núm mát-xa nhỏ, mềm mịn, tạo cảm giác dễ chịu và thích thú khi bé sử dụng. Thiết kế ngộ nghĩnh: Hình động vật với tay cầm Sản phẩm được thiết kế dạng hình các con vật đồ dùng ngộ nghĩnh, đặc biệt có tay cầm giúp bé dễ cầm nắm và chơi đùa. Độ đàn hồi tốt: Bền đẹp và dễ sử dụng Miếng gặm nướu có độ đàn hồi tốt, dễ dàng uốn dẻo, bền đẹp mà không gây ảnh hưởng đến sức khỏe cho bé. Cảm giác mát lạnh: Hỗ trợ giảm đau Trước khi cho bé sử dụng, bạn có thể cho vào tủ lạnh tạo cảm giác mát lạnh, giúp bé thích thú và giảm đau nướu. Hướng dẫn sử dụng Rửa sạch sản phẩm trước khi cho bé sử dụng. Để sản phẩm trong tủ lạnh trước khi cho bé sử dụng để tạo cảm giác mát lạnh. Giám sát bé khi sử dụng để đảm bảo an toàn. Hướng dẫn bảo quản Bảo quản sản phẩm ở nơi khô ráo, thoáng mát. Không sử dụng nước sôi để khử trùng sản phẩm. Thường xuyên kiểm tra sản phẩm để đảm bảo không bị hư hỏng. Lưu ý Sản phẩm chỉ dành cho bé trong giai đoạn mọc răng. Khi sản phẩm có dấu hiệu hư hỏng, không nên cho bé sử dụng. …"

In [145]:
kxd_list = ["không xác định","null"]

s_new = pl.col("description_new").cast(pl.Utf8).str.to_lowercase().str.strip_chars()
s_old = pl.col("description").cast(pl.Utf8).str.to_lowercase().str.strip_chars()

new_has_text = pl.col("description_new").is_not_null() & (s_new.str.len_chars() > 0) & (~s_new.is_in(kxd_list))
old_has_text = pl.col("description").is_not_null() & (s_old.str.len_chars() > 0) & (~s_old.is_in(kxd_list))

counts_value = (
    df_cleaned.select(
        (new_has_text & old_has_text).sum().alias("both_present"),
        (new_has_text & ~old_has_text).sum().alias("only_description_new"),
        (~new_has_text & old_has_text).sum().alias("only_description"),
        (~new_has_text & ~old_has_text).sum().alias("neither_present"),
    )
    .with_columns(
        total = pl.sum_horizontal(pl.all())
    )
    .with_columns(
        *[(pl.col(c) / pl.col("total") * 100).round(2).alias(f"{c}_pct")
          for c in ["both_present","only_description_new","only_description","neither_present"]]
    )
)
print(counts_value)

shape: (1, 9)
┌──────────────┬──────────────────────┬──────────────────┬─────────────────┬───────┬──────────────────┬──────────────────────────┬──────────────────────┬─────────────────────┐
│ both_present ┆ only_description_new ┆ only_description ┆ neither_present ┆ total ┆ both_present_pct ┆ only_description_new_pct ┆ only_description_pct ┆ neither_present_pct │
│ ---          ┆ ---                  ┆ ---              ┆ ---             ┆ ---   ┆ ---              ┆ ---                      ┆ ---                  ┆ ---                 │
│ u32          ┆ u32                  ┆ u32              ┆ u32             ┆ u32   ┆ f64              ┆ f64                      ┆ f64                  ┆ f64                 │
╞══════════════╪══════════════════════╪══════════════════╪═════════════════╪═══════╪══════════════════╪══════════════════════════╪══════════════════════╪═════════════════════╡
│ 6308         ┆ 3194                 ┆ 2395             ┆ 15435           ┆ 27332 ┆ 23.08            ┆ 11

**Độ phủ hai cột mô tả: `description_new` vs `description`**

**Bảng tóm tắt thông tin:**

| Trường hợp | Số dòng | Tỷ lệ (%) |
|---|---:|---:|
| **Cả hai đều có** (`both_present`) | **6,308** | **23.08** |
| **Chỉ có `description_new`** | **3,194** | **11.69** |
| **Chỉ có `description`** | **2,395** | **8.76** |
| **Không có cả hai** (`neither_present`) | **15,435** | **56.47** |
| **Tổng** | **27,332** | **100.00** |

- Hàng **có `description_new`** = 6,308 + 3,194 = **9,502** (**34.77%**).  
  ⟶ **Thiếu `description_new`** ≈ **17,830**.
- Hàng **có `description`** = 6,308 + 2,395 = **8,703** (**31.84%**).  
  ⟶ **Thiếu `description`** ≈ **18,629**.
- Hàng **có ít nhất một** mô tả = 27,332 − 15,435 = **11,897** (**43.53%**).  


### ⟶ **Dự định tiếp theo**: Gộp description và description_new lại để bổ sung dữ liệu cho nhau, giảm chiều dữ liệu



#### **BÀI TOÁN MỚI CẦN GIẢI QUYẾT**: Nên gộp `description` vào `description_new` hay ngược lại?  


In [146]:
import polars as pl
import re

# ===== 1) Cấu hình =====
LEVEL = "word"   # "word" hoặc "char"
N = 2            # n-gram: 1/2/3... ; ví dụ word-bigram = 2

# Các giá trị được coi là "thiếu"
PLACEHOLDERS = [
    "", "-", "--",
    "không xác định", "khong xac dinh", "không xac dinh",
    "không rõ", "chưa rõ", "không có thông tin",
    "n/a", "na", "none", "null", "unk", "unknown"
]

# ===== 2) Hàm tiện ích =====
def tokenize(s: str):
    return re.findall(r"\w+", s.lower())

def ngrams(tokens, n=2):
    if len(tokens) < n: 
        return []
    return [" ".join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def build_set(s: str, level="word", n=2):
    if not s:
        return set()
    if level == "char":
        s_norm = re.sub(r"\s+", " ", s.lower()).strip()
        if len(s_norm) < n:
            return set()
        return set(s_norm[i:i+n] for i in range(len(s_norm)-n+1))
    else:
        toks = tokenize(s)
        return set(toks) if n == 1 else set(ngrams(toks, n))

def contain_ratio_sets(A_set: set, B_set: set) -> float:
    return 0.0 if not A_set else len(A_set & B_set) / len(A_set)

def contain_pair(a: str, b: str, level="word", n=2):
    A = build_set(a, level=level, n=n)
    B = build_set(b, level=level, n=n)
    r = contain_ratio_sets(A, B)  # Contain(A|B)
    p = contain_ratio_sets(B, A)  # Contain(B|A)
    f1 = 0.0 if (p + r) == 0 else 2 * p * r / (p + r)
    return (r, p, f1, len(A), len(B))

# Chuẩn hoá text để so đối chiếu placeholder
def norm_expr(colname: str):
    return (
        pl.col(colname).cast(pl.Utf8)
        .str.to_lowercase()
        .str.replace_all(r"\s+", " ")
        .str.strip_chars(" .,-:;—–_/\\|\"'`")  # bỏ dấu câu ở đầu/cuối
        .str.strip_chars()                     # bỏ khoảng trắng thừa
    )

def is_missing(colname: str):
    n = norm_expr(colname)
    return pl.col(colname).is_null() | n.is_in(PLACEHOLDERS)

# ===== 3) Mask both_present với định nghĩa "thiếu" mới =====
miss_desc     = is_missing("description")
miss_desc_new = is_missing("description_new")
both_present  = (~miss_desc) & (~miss_desc_new)

# Báo cáo thiếu dữ liệu (fraction)
null_report = df_cleaned.select([
    (miss_desc & miss_desc_new).mean().alias("both_null_frac"),
    (miss_desc & (~miss_desc_new)).mean().alias("desc_null_only_frac"),
    ((~miss_desc) & miss_desc_new).mean().alias("desc_new_null_only_frac"),
    both_present.mean().alias("both_present_frac"),
])
print("=== Null report (fraction) — placeholders treated as null ===")
print(null_report)

# ===== 4) Tính containment chỉ trên both_present =====
both_df = df_cleaned.filter(both_present)

res = both_df.with_columns([
    pl.struct(["description", "description_new"]).map_elements(
        lambda row: contain_pair(row["description"], row["description_new"], level=LEVEL, n=N)[0]
    ).alias("contain_desc_in_descnew"),   # recall: A|B

    pl.struct(["description", "description_new"]).map_elements(
        lambda row: contain_pair(row["description"], row["description_new"], level=LEVEL, n=N)[1]
    ).alias("contain_descnew_in_desc"),   # precision: B|A

    pl.struct(["description", "description_new"]).map_elements(
        lambda row: contain_pair(row["description"], row["description_new"], level=LEVEL, n=N)[2]
    ).alias("contain_f1"),
])

summary = res.select([
    pl.len().alias("n_both_present"),
    pl.col("contain_desc_in_descnew").mean().alias("mean_contain(A|B)"),
    pl.col("contain_descnew_in_desc").mean().alias("mean_contain(B|A)"),
    pl.col("contain_f1").mean().alias("mean_contain_F1"),
    pl.col("contain_desc_in_descnew").quantile(0.5).alias("median_contain(A|B)"),
    pl.col("contain_desc_in_descnew").quantile(0.9).alias("p90_contain(A|B)"),
])
print(f"=== Containment summary (LEVEL={LEVEL}, N={N}) — after placeholder filtering ===")
print(summary)

# (Tuỳ chọn) kiểm tra vài dòng F1 cao
print(res.select(["description", "description_new", "contain_desc_in_descnew", "contain_descnew_in_desc", "contain_f1"])
        .sort("contain_f1", descending=True)
        .head(10))


=== Null report (fraction) — placeholders treated as null ===
shape: (1, 4)
┌────────────────┬─────────────────────┬─────────────────────────┬───────────────────┐
│ both_null_frac ┆ desc_null_only_frac ┆ desc_new_null_only_frac ┆ both_present_frac │
│ ---            ┆ ---                 ┆ ---                     ┆ ---               │
│ f64            ┆ f64                 ┆ f64                     ┆ f64               │
╞════════════════╪═════════════════════╪═════════════════════════╪═══════════════════╡
│ 0.564723       ┆ 0.116859            ┆ 0.087626                ┆ 0.230792          │
└────────────────┴─────────────────────┴─────────────────────────┴───────────────────┘
=== Containment summary (LEVEL=word, N=2) — after placeholder filtering ===
shape: (1, 6)
┌────────────────┬───────────────────┬───────────────────┬─────────────────┬─────────────────────┬──────────────────┐
│ n_both_present ┆ mean_contain(A|B) ┆ mean_contain(B|A) ┆ mean_contain_F1 ┆ median_contain(A|B) ┆ p90_cont

**mean Contain(A|B) = 0.623**
- Trung bình 62.3% bigram trong description cũng có trong description_new.
- Khi có dữ liệu thật, description_new bao phủ phần lớn nội dung của description.

**mean Contain(B|A) = 0.324**
- Chỉ 32.4% bigram trong description_new xuất hiện trong description.
-  description_new giàu/chi tiết hơn, có nhiều cụm mới (vd. “Tên sản phẩm/Thương hiệu/Xuất xứ/Kích thước…”) mà description không có.
-  Đây là lý do precision thấp.

**mean F1 = 0.405**
- Trung bình độ khớp hai chiều chỉ ở mức vừa (hàm hòa hợp của hai contain). Bị kéo xuống bởi B|A thấp.

**median Contain(A|B) = 0.620; p90 = 0.807**
- 50% cặp có ≥62% nội dung description được bao bởi description_new; top 10% đạt ≥80.7%.
- Phân bố khá lệch: nhiều cặp khớp tốt, nhưng cũng có không ít cặp lệch mạnh (kéo mean xuống).

### Xử lý  description và description_new

### Gộp 2 cột description và description_new

- Sau khi đã xác định các giá trị thiếu chỉ gồm null hoặc "Không xác định" thì chỉ cần kiểm tra missing(x) = is_null(x) OR lower(strip(x)) == "Không xác định" 
- Để nhận diện thiếu, tránh copy nhầm "Không xác định"

**Logic gộp**
1. Nếu des_new không missing => lấy des_new
2. Nếu des_new missing nhưng des không missing => lấy des 
3. Nếu cả hai missing => Để NULL
4. Thêm cột description_source = new / old_as_fill / missing để audit

In [147]:
pl.Config.set_tbl_rows(20) 
pl.Config.set_tbl_cols(20) 
pl.Config.set_tbl_width_chars(150)
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [148]:
# --- 1️⃣ Định nghĩa hàm phát hiện "thiếu dữ liệu" ---
def is_missing(col: str) -> pl.Expr:
    c = pl.col(col)
    return c.is_null() | (
        c.cast(pl.Utf8)
         .str.to_lowercase()
         .str.strip_chars()
         .is_in(["không xác định", "null", "none", ""])
    )

# --- 2️⃣ Tạo mask ---
miss_new = is_missing("description_new")
miss_old = is_missing("description")

# --- 3️⃣ Gộp hai mô tả và đánh dấu nguồn ---
df_cleaned = df_cleaned.with_columns([
    pl.when(~miss_new).then(pl.col("description_new"))
      .when(~miss_old).then(pl.col("description"))
      .otherwise(None)
      .alias("description_merge"),

    pl.when(~miss_new).then(pl.lit("from_new"))
     .when(~miss_old).then(pl.lit("from_old"))
     .otherwise(pl.lit("missing"))
     .alias("description_source")
])

# --- 4️⃣ Báo cáo tỷ lệ merge ---
merge_report = (
    df_cleaned.select(
        pl.count().alias("total"),
        (pl.col("description_source") == "from_new").sum().alias("from_new"),
        (pl.col("description_source") == "from_old").sum().alias("filled_from_old"),
        (pl.col("description_source") == "missing").sum().alias("still_missing"),
    )
    .with_columns([
        (pl.col("from_new") / pl.col("total") * 100).round(2).alias("from_new_pct"),
        (pl.col("filled_from_old") / pl.col("total") * 100).round(2).alias("filled_from_old_pct"),
        (pl.col("still_missing") / pl.col("total") * 100).round(2).alias("still_missing_pct"),
    ])
)

print("=== 🧩 Merge report (description + description_new) ===")
print(merge_report)

# --- 5️⃣ Xem vài dòng mẫu được fill từ mô tả cũ ---
print("\n=== 🔍 Sample rows (filled_from_old) ===")
print(
    df_cleaned.filter(pl.col("description_source") == "from_old")
              .select(["description_new", "description", "description_merge"])
              .head(5)
)



=== 🧩 Merge report (description + description_new) ===
shape: (1, 7)
┌───────┬──────────┬─────────────────┬───────────────┬──────────────┬─────────────────────┬───────────────────┐
│ total ┆ from_new ┆ filled_from_old ┆ still_missing ┆ from_new_pct ┆ filled_from_old_pct ┆ still_missing_pct │
│ ---   ┆ ---      ┆ ---             ┆ ---           ┆ ---          ┆ ---                 ┆ ---               │
│ u32   ┆ u32      ┆ u32             ┆ u32           ┆ f64          ┆ f64                 ┆ f64               │
╞═══════╪══════════╪═════════════════╪═══════════════╪══════════════╪═════════════════════╪═══════════════════╡
│ 27332 ┆ 9502     ┆ 2395            ┆ 15435         ┆ 34.77        ┆ 8.76                ┆ 56.47             │
└───────┴──────────┴─────────────────┴───────────────┴──────────────┴─────────────────────┴───────────────────┘

=== 🔍 Sample rows (filled_from_old) ===
shape: (5, 3)
┌─────────────────┬─────────────────────────────────────────────────────────────────┬───────

C:\Users\PC\AppData\Local\Temp\ipykernel_10564\9102509.py:31: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total"),


Sau khi đã kiểm tra xong và thấy mọi thứ đã hợp lý, tôi sẽ drop các cột cũ không cần dùng đến (description, description_new, description_source), chỉ giữ lại cột description_merge

In [149]:
# --- Xóa cột cũ để giữ dataset gọn gàng ---
df_cleaned = df_cleaned.drop(["description", "description_new", "description_source"])

In [150]:
df_cleaned.shape
df_cleaned.head(5)

item_id,price,category_l1,category,brand,gender_target,age_group,item_type,color,size,origin,material,sale_status,price_capped,description_merge
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,i32,f64,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,99000.0,"""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…"
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,69000.0,null
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,45000.0,"""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) C…"
"""0020010000094""",401000.0000,"""Tã""","""Merries_Sơ Sinh""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt, hạt thấm hút, màng polyolefin, cao su tự nhiên, polyurethane, keo dín…",0,401000.0,"""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của…"
"""0020010000098""",401000.0000,"""Tã""","""Merries_Tã Quần""","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Nhật Bản, Nhật Bản""","""Giấy, bột giấy, vải không dệt, hạt thấm hút, màng polyolefin, cao su tự nhiên, polyurethane, keo dín…",0,401000.0,"""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm dành cho bé từ 6-11kg đến từ thương hiệu uy tín M…"


In [13]:
#df_cleaned.filter(pl.col("item_id") == "4690000000001")


## Xử lý NULL sau khi phân tích

Như đã đề cặp ở trên,sau khi EDA, tôi đã thấy rằng, cột description và description_new có độ tương quan khá cao với các cột: brand, age_group, item_type, color, origin, material. 
Mặc khác, các cột:  brand, age_group, item_type, color, origin, material đều là những feature quan trọng cho bài toán dự đoán mua hàng. Vì vậy,sau khi đã gộp 2 cột description và description_new thành cột description_merge, chúng tôi dự định sẽ tận dụng cột dữ liệu này để điền các dữ liệu trống cho các cột: brand, age_group, item_type, color, origin, material.

### size và age_group

##### Bucket hóa age_group

In [151]:
df_cleaned.select(pl.col("age_group").unique())

age_group
str
"""Từ 12Y"""
"""2Y-6Y"""
"""1Y-2Y"""
"""11Y-12Y"""
"""Trên 6M"""
"""Từ 7Y"""
"""0-18M"""
"""2Y-10Y"""
"""[""Từ 3M"", ""Từ 6M""]"""


Do biến age_group quá nhiều giá trị rời rạc (Có giá trị theo tháng (M), Có giá trị theo năm (Y),Có giá trị pha trộn cả hai đơn vị → 0-5Y (gộp nhiều năm), 18M-36M (tháng nhưng tới 3 năm)), vì vậy, tôi đã tiến hành chuẩn hóa cột dữ liệu này: 
Gom các giá trị chi tiết về độ tuổi (như 0-3M, 6-12M, Từ 1Y, 3-5Y, …)
→ thành nhóm (bucket) có ý nghĩa tổng quát hơn, như:
- 0-6M (sơ sinh)
- 6-12M (bé tập bò)
- 1-2Y
- 2-4Y
- .>4Y
- ...
- Mẹ

Mục tiêu chính là:
- Chuẩn hóa dữ liệu không đồng nhất,
- Giảm số lượng giá trị rời rạc (unique values),
- Và giúp mô hình ML học được xu hướng tổng thể thay vì bị nhiễu bởi chi tiết lẻ.

In [153]:
import re, json

# ── Chuẩn hoá chuỗi để parse ──────────────────────────────────────────────
def _norm_age(s: str) -> str:
    s = s.lower().strip()
    s = (s.replace("tháng", "m").replace("thang", "m")
           .replace("tuổi", "y").replace("tuoi", "y")
           .replace("từ", "tu").replace("trên", "tren")
           .replace("–", "-").replace("—", "-").replace("−", "-")
           .replace(" ", ""))
    # toddler 'T' -> 'y' (2T = 2y)
    s = re.sub(r"(?<=\d)t\b", "y", s)
    # gom newborn
    s = (s.replace("sơsinh", "nb")
           .replace("sosinh", "nb")
           .replace("newborn", "nb"))
    return s

# ── Parse về (lo, hi) theo THÁNG; cover cả "0-5Y", "0-4Y", ... ────────────
def _parse_to_months(val: str | None):
    if val is None:
        return (None, None)

    raw = str(val).strip()

    # list-string JSON: ["Từ 0M","Từ 3M",...]
    if raw.lstrip().startswith("["):
        try:
            items = json.loads(raw)
            lows, highs = [], []
            for it in items:
                lo, hi = _parse_to_months(it)
                if lo is not None: lows.append(lo)
                if hi is not None: highs.append(hi)
            if lows or highs:
                return (min(lows) if lows else None, max(highs) if highs else None)
        except Exception:
            pass  # rơi xuống parse thường

    s = _norm_age(raw)

    # giữ nguyên "Mẹ"
    if s in {"me", "mẹ"}:
        return ("Mẹ", "Mẹ")  # dùng sentinel để lát nữa trả về "Mẹ"

    # newborn
    if s == "nb":
        return (0, 3)

    # 1) RANGE với đơn vị tuỳ chọn ở MỖI VẾ
    m = re.fullmatch(r"(\d+)(m|y)?-(\d+)(m|y)?", s)
    if m:
        a, u1, b, u2 = m.groups()
        if u1 is None and u2 is None:
            pass
        else:
            if u1 is None: u1 = u2
            if u2 is None: u2 = u1
            a = int(a) * (1 if u1 == "m" else 12)
            b = int(b) * (1 if u2 == "m" else 12)
            if a > b: a, b = b, a
            return (a, b)

    # 2) SINGLE: 9m, 2y
    m = re.fullmatch(r"(\d+)(m|y)", s)
    if m:
        a, u = m.groups()
        a = int(a) * (1 if u == "m" else 12)
        return (a, a)

    # 3) THRESHOLD: tu6m, tren2y
    m = re.fullmatch(r"(tu|tren)(\d+)(m|y)", s)
    if m:
        v = int(m.group(2)) * (1 if m.group(3) == "m" else 12)
        return (v, None)

    return (None, None)

# ── Map (lo,hi) -> primary bucket ─────────────────────────────────────────
def _bucket_from_months(lo, hi):
    if lo == "Mẹ" and hi == "Mẹ":
        return "Mẹ"
    if lo is None and hi is None:
        return None
    mid = lo if (hi is None or isinstance(lo, str)) else (lo + hi) / 2.0
    if isinstance(mid, str):  # đề phòng "Mẹ"
        return "Mẹ"
    if mid < 6:   return "0-6M"
    if mid < 12:  return "6-12M"
    if mid < 24:  return "1-2Y"
    if mid < 48:  return "2-4Y"
    return ">4Y"

def age_to_primary_bucket(val):
    lo, hi = _parse_to_months(val)
    return _bucket_from_months(lo, hi)

# ── Tạo cột bucket ────────────────────────────────────────────────────────
df_cleaned = df_cleaned.with_columns(
    pl.col("age_group")
      .map_elements(age_to_primary_bucket, return_dtype=pl.Utf8)
      .alias("age_bucket_primary")
)

# ── (Tuỳ chọn) xem phân bố ───────────────────────────────────────────────
print(df_cleaned.select(pl.col("age_bucket_primary").value_counts()))


shape: (7, 1)
┌────────────────────┐
│ age_bucket_primary │
│ ---                │
│ struct[2]          │
╞════════════════════╡
│ {"6-12M",3073}     │
│ {null,15812}       │
│ {"Mẹ",150}         │
│ {"0-6M",1635}      │
│ {">4Y",1746}       │
│ {"2-4Y",2753}      │
│ {"1-2Y",2163}      │
└────────────────────┘


In [154]:
df_cleaned.head(3)

item_id,price,category_l1,category,brand,gender_target,age_group,item_type,color,size,origin,material,sale_status,price_capped,description_merge,age_bucket_primary
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,i32,f64,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,99000.0,"""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…",null
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,69000.0,null,"""2-4Y"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,45000.0,"""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) C…",null


In [155]:
df_cleaned = df_cleaned.drop("age_group")
df_cleaned.head(3)

item_id,price,category_l1,category,brand,gender_target,item_type,color,size,origin,material,sale_status,price_capped,description_merge,age_bucket_primary
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,i32,f64,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Núm ty Dr Brown""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,99000.0,"""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…",null
"""0010290040150""",69000.0000,"""Thời trang""","""Bộ quần áo bé gái""","""Con Cưng""","""Bé Gái""","""Bộ quần áo""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,69000.0,null,"""2-4Y"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""Gặm nướu khác""","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""",0,45000.0,"""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) C…",null


##### Dùng size để fill cho age_group

In [156]:
df_cleaned.select(pl.col("size").unique())

size
str
"""0-3M"""
"""1"""
"""Từ 2 tuổi"""
"""17"""
"""M(7-12kg) - 76 miếng"""
"""XXL (>15kg) 54 miếng"""
"""2-3Y"""
"""9 tháng"""
"""26"""


Phân tích dữ liệu của cột size:
- Cột size của bạn đang trộn lẫn nhiều loại thông tin khác nhau, không chỉ là “kích cỡ vật lý” mà còn ẩn chứa thông tin tuổi
- Trong cột size hiện tại đang có thông tin ở 3 dạng khác nhau:
    + Dạng độ tuổi (tháng, năm): "6-9M", "12-24M", "1-2 tuổi"
    + Dạng kích cỡ vật lý: "110", "16", "20", "36", "28*36cm"
    + Dạng mô tả khác: "M(6-11kg) - 64 miếng", "XXL"

In [157]:
# =====================================================
# 🧩 Fill cột age_bucket_primary dựa trên thông tin trong cột size
# =====================================================

from functools import lru_cache

# =====================================================
# 0) Tham số
# =====================================================
MAX_MONTHS = 216  # Giới hạn tối đa (18 năm)

# =====================================================
# 1) Chuẩn hoá & Parse chuỗi tuổi (size dạng tuổi)
# =====================================================
def _norm_age(s: str) -> str:
    """Chuẩn hoá text về dạng dễ nhận diện (vd: '6M', '2Y', 'tu6m', 'sosinh'...)"""
    s = s.lower().strip()
    s = (
        s.replace("tháng", "m").replace("thang", "m")
         .replace("tuổi", "y").replace("tuoi", "y")
         .replace("từ", "tu").replace("trên", "tren")
         .replace("–", "-").replace("—", "-").replace("−", "-")
         .replace(" ", "")
    )
    s = re.sub(r"(?<=\d)t\b", "y", s)  # toddler 2T → 2Y
    s = (
        s.replace("sơsinh", "nb")
         .replace("sosinh", "nb")
         .replace("newborn", "nb")
    )
    return s


def parse_age_text_to_months(val: str | None):
    """Chuyển text thành (lo, hi) theo tháng."""
    if val is None:
        return (None, None)

    raw = str(val).strip()
    if raw == "" or raw.lower() in ["không xác định", "none", "null", "na"]:
        return (None, None)

    # list JSON: ["Từ 0M","Từ 3M",...]
    if raw.lstrip().startswith("["):
        try:
            items = json.loads(raw)
            lows, highs = [], []
            for it in items:
                lo, hi = parse_age_text_to_months(it)
                if lo is not None: lows.append(lo)
                if hi is not None: highs.append(hi)
            if lows or highs:
                return (min(lows) if lows else None, max(highs) if highs else None)
        except Exception:
            pass

    s = _norm_age(raw)

    if s in {"me", "mẹ"}:
        return (None, None)
    if s == "nb":
        return (0, 3)

    # RANGE (0-5y, 6m-12m, 1y-3y, ...)
    m = re.fullmatch(r"(\d+)(m|y)?-(\d+)(m|y)?", s)
    if m:
        a, u1, b, u2 = m.groups()
        if u1 or u2:
            if u1 is None: u1 = u2
            if u2 is None: u2 = u1
            a = int(a) * (1 if u1 == "m" else 12)
            b = int(b) * (1 if u2 == "m" else 12)
            if a > b: a, b = b, a
            return (a, b)

    # SINGLE (9m, 2y)
    m = re.fullmatch(r"(\d+)(m|y)", s)
    if m:
        a, u = m.groups()
        a = int(a) * (1 if u == "m" else 12)
        return (a, a)

    # THRESHOLD (tu6m, tren2y)
    m = re.fullmatch(r"(tu|tren)(\d+)(m|y)", s)
    if m:
        v = int(m.group(2)) * (1 if m.group(3) == "m" else 12)
        return (v, None)

    return (None, None)


def primary_bucket_from_months(lo: int | None, hi: int | None) -> str | None:
    """Quy đổi khoảng tuổi (tháng) thành bucket chuẩn."""
    if lo is None and hi is None:
        return None
    mid = lo if hi is None else (lo + hi) / 2.0
    if mid < 6:   return "0-6M"
    if mid < 12:  return "6-12M"
    if mid < 24:  return "1-2Y"
    if mid < 48:  return "2-4Y"
    return ">4Y"

# =====================================================
# 2) RULE: Nhận diện size có chứa thông tin tuổi
# =====================================================
def size_rules_extract_months(size_val: str | None):
    """Nếu size chứa thông tin tuổi -> trả về (lo, hi)"""
    if size_val is None:
        return (None, None)
    lo, hi = parse_age_text_to_months(size_val)
    if lo is None and hi is None:
        return (None, None)
    l = lo if lo is not None else 0
    h = hi if hi is not None else MAX_MONTHS
    if l < 0 or h < 0 or h > MAX_MONTHS or (hi and lo and lo > hi):
        return (None, None)
    return (lo, hi)

# =====================================================
# 3) Tạo bảng mapping từ size → age bucket
# =====================================================
size_unique = (
    df_cleaned.select(pl.col("size").cast(pl.Utf8))
              .unique()
              .drop_nulls()
)

size_map_rows = []
for s in size_unique.get_column("size").to_list():
    lo, hi = size_rules_extract_months(s)
    source = "size_rule" if (lo is not None or hi is not None) else None
    bucket = primary_bucket_from_months(lo, hi) if source else None
    size_map_rows.append((s, lo, hi, bucket, source))

size_map = pl.DataFrame(
    size_map_rows,
    schema={
        "size": pl.Utf8,
        "size_lo_m": pl.Int64,
        "size_hi_m": pl.Int64,
        "size_age_bucket": pl.Utf8,
        "age_fill_source": pl.Utf8,
    },
)

# =====================================================
# 4) Join mapping vào df_cleaned
# =====================================================
df_with_size_age = df_cleaned.join(size_map, on="size", how="left")

# =====================================================
# 5) Điền bổ sung cột age_bucket_primary còn thiếu
# =====================================================
df_filled = df_with_size_age.with_columns(
    pl.when(pl.col("age_bucket_primary").is_null() & pl.col("size_age_bucket").is_not_null())
      .then(pl.col("size_age_bucket"))
      .otherwise(pl.col("age_bucket_primary"))
      .alias("age_bucket_primary_filled")
)

# =====================================================
# 6) Báo cáo kết quả coverage
# =====================================================
coverage_before = df_with_size_age.select(pl.col("age_bucket_primary").is_not_null().sum()).item()
coverage_after  = df_filled.select(pl.col("age_bucket_primary_filled").is_not_null().sum()).item()
total_rows      = df_filled.height

print({
    "total_rows": total_rows,
    "coverage_before": coverage_before,
    "coverage_after": coverage_after,
    "delta": coverage_after - coverage_before
})

# Gán lại biến chính
df_cleaned = df_filled


{'total_rows': 27332, 'coverage_before': 11520, 'coverage_after': 12359, 'delta': 839}


C:\Users\PC\AppData\Local\Temp\ipykernel_10564\3809450951.py:135: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  size_map = pl.DataFrame(


In [158]:
def show(df: pl.DataFrame, title: str, n: int = 20):
    print(f"\n=== {title} (show up to {n}) ===")
    cols = [c for c in [
        "category_l1", "product_id", "sku", "title",
        "size", "age_group",
        "size_lo_m", "size_hi_m", "size_age_bucket",
        "age_bucket_primary", "age_bucket_primary_filled",
        "age_fill_source", "age_llm_conf"
    ] if c in df.columns]
    print(df.select(cols).sample(n=min(n, df.height), seed=42) if df.height else "— No rows —")

# 1) Các dòng đã được FILL từ size
filled_rows = df_filled.filter(
    pl.col("age_bucket_primary").is_null() &
    pl.col("age_bucket_primary_filled").is_not_null()
)
show(filled_rows, "ĐÃ ĐIỀN bucket từ size", n=20)

# 2) Bất đồng giữa age_group bucket và size bucket (để kiểm tra)
disagree = df_filled.filter(
    pl.col("age_bucket_primary").is_not_null() &
    pl.col("size_age_bucket").is_not_null() &
    (pl.col("age_bucket_primary") != pl.col("size_age_bucket"))
)
show(disagree, "KHÁC NHAU giữa bucket từ age_group vs size (chỉ xem)", n=20)

# 3) Cả hai đều thiếu
both_missing = df_filled.filter(
    pl.col("age_bucket_primary").is_null() &
    pl.col("size_age_bucket").is_null()
)
show(both_missing, "CẢ HAI THIẾU (không có bucket)", n=20)

# 4) "Mẹ" giữ nguyên
adult_kept = df_filled.filter(pl.col("age_bucket_primary") == "Mẹ")
show(adult_kept, '"Mẹ" được giữ nguyên', n=10)

# 5) Đếm nhanh
print("\n=== COUNTS ===")
counts = pl.DataFrame({
    "metric": [
        "total_rows",
        "filled_from_size",
        "have_any_bucket_after_fill",
        "both_missing",
        "disagree_age_vs_size"
    ],
    "value": [
        df_filled.height,
        filled_rows.height,
        df_filled.filter(pl.col("age_bucket_primary_filled").is_not_null()).height,
        both_missing.height,
        disagree.height
    ]
})
print(counts)


=== ĐÃ ĐIỀN bucket từ size (show up to 20) ===
shape: (20, 8)
┌─────────────┬──────────┬───────────┬───────────┬─────────────────┬────────────────────┬───────────────────────────┬─────────────────┐
│ category_l1 ┆ size     ┆ size_lo_m ┆ size_hi_m ┆ size_age_bucket ┆ age_bucket_primary ┆ age_bucket_primary_filled ┆ age_fill_source │
│ ---         ┆ ---      ┆ ---       ┆ ---       ┆ ---             ┆ ---                ┆ ---                       ┆ ---             │
│ str         ┆ str      ┆ i64       ┆ i64       ┆ str             ┆ str                ┆ str                       ┆ str             │
╞═════════════╪══════════╪═══════════╪═══════════╪═════════════════╪════════════════════╪═══════════════════════════╪═════════════════╡
│ Thời trang  ┆ 2-3Y     ┆ 24        ┆ 36        ┆ 2-4Y            ┆ null               ┆ 2-4Y                      ┆ size_rule       │
│ Thời trang  ┆ 9 tháng  ┆ 9         ┆ 9         ┆ 6-12M           ┆ null               ┆ 6-12M                     ┆ siz

##### dùng description_merge để fill thêm cho những dòng còn thiếu age_bucket_primary_filled

In [159]:
# === Cell: Trích tuổi từ description_merged và điền bổ sung ===

df = df_cleaned  # dùng DF hiện tại

# -- helper: map mid(tháng) -> nhãn bucket (dùng pl.lit cho LITERAL) --
def bucket_from_mid_expr(mid: pl.Expr) -> pl.Expr:
    return (
        pl.when(mid.is_null()).then(pl.lit(None))
         .when(mid < 6).then(pl.lit("0-6M"))
         .when(mid < 12).then(pl.lit("6-12M"))
         .when(mid < 24).then(pl.lit("1-2Y"))
         .when(mid < 48).then(pl.lit("2-4Y"))
         .otherwise(pl.lit(">4Y"))
    )

# (an toàn) nếu có cột bucket cũ dùng en-dash, đưa về ASCII hyphen
if "age_bucket_primary_filled" in df.columns:
    df = df.with_columns(
        pl.col("age_bucket_primary_filled").cast(pl.Utf8)
          .str.replace_all("–|—", "-")
          .alias("age_bucket_primary_filled")
    )

# nếu chưa có cột primary, tạo từ age_month_mid (nếu có), else để None
if "age_bucket_primary_filled" not in df.columns:
    df = df.with_columns(
        bucket_from_mid_expr(pl.col("age_month_mid")).alias("age_bucket_primary_filled")
        if "age_month_mid" in df.columns else pl.lit(None).alias("age_bucket_primary_filled")
    )

# 1) chuẩn hoá mô tả để parse (không đổi giá trị gốc)
df = df.with_columns(
    pl.col("description_merge").cast(pl.Utf8)
      .str.to_lowercase()
      .str.replace_all('"', '').str.replace_all('“|”', '')
      .str.replace_all(r"\s+", " ").str.strip_chars()
      .alias("_desc_norm")
)

# 2) trích tuổi (tháng) từ mô tả
d_range_m_lo = pl.col("_desc_norm").str.extract(r'(\d{1,2})\s*[-–—]\s*(\d{1,2})\s*(?:m|mo|tháng)\b', 1).cast(pl.Int64)
d_range_m_hi = pl.col("_desc_norm").str.extract(r'(\d{1,2})\s*[-–—]\s*(\d{1,2})\s*(?:m|mo|tháng)\b', 2).cast(pl.Int64)
d_range_y_lo = pl.col("_desc_norm").str.extract(r'(\d{1,2})\s*[-–—]\s*(\d{1,2})\s*(?:y|yr|years|tuổi|năm)\b', 1).cast(pl.Int64) * 12
d_range_y_hi = pl.col("_desc_norm").str.extract(r'(\d{1,2})\s*[-–—]\s*(\d{1,2})\s*(?:y|yr|years|tuổi|năm)\b', 2).cast(pl.Int64) * 12
d_lb_m       = pl.col("_desc_norm").str.extract(r'(?:từ|>=|trên)\s*(\d{1,2})\s*(?:m|mo|tháng)\b', 1).cast(pl.Int64)
d_lb_y       = pl.col("_desc_norm").str.extract(r'(?:từ|>=|trên)\s*(\d{1,2})\s*(?:y|yr|years|tuổi|năm)\b', 1).cast(pl.Int64) * 12
d_single_m   = pl.col("_desc_norm").str.extract(r'\b(\d{1,2})\s*(?:m|mo|tháng)\b', 1).cast(pl.Int64)
d_single_y   = pl.col("_desc_norm").str.extract(r'\b(\d{1,2})\s*(?:y|yr|years|tuổi|năm)\b', 1).cast(pl.Int64) * 12
d_nb_lo      = pl.when(pl.col("_desc_norm").str.contains(r'\b(nb|newborn|sơ sinh)\b')).then(pl.lit(0)).otherwise(None)
d_nb_hi      = pl.when(pl.col("_desc_norm").str.contains(r'\b(nb|newborn|sơ sinh)\b')).then(pl.lit(3)).otherwise(None)

df = df.with_columns([
    pl.coalesce([d_range_m_lo, d_range_y_lo, d_nb_lo, d_lb_m, d_lb_y, d_single_m, d_single_y]).alias("desc_age_month_lo"),
    pl.coalesce([d_range_m_hi, d_range_y_hi, d_nb_hi]).alias("desc_age_month_hi"),
]).with_columns([
    pl.when(pl.col("desc_age_month_lo").is_not_null() & pl.col("desc_age_month_hi").is_not_null())
      .then((pl.col("desc_age_month_lo") + pl.col("desc_age_month_hi")) / 2)
      .when(pl.col("desc_age_month_lo").is_not_null())
      .then(pl.col("desc_age_month_lo").cast(pl.Float64))
      .otherwise(None)
      .alias("desc_age_month_mid"),
])

# 3) bucket mô tả theo chuẩn ASCII (dùng pl.lit trong bucket_from_mid_expr)
df = df.with_columns(
    bucket_from_mid_expr(pl.col("desc_age_month_mid")).alias("desc_age_bucket")
)

# 4) confidence đơn giản
df = df.with_columns(
    pl.when(pl.col("desc_age_month_lo").is_not_null() & pl.col("desc_age_month_hi").is_not_null()).then(pl.lit(0.95))
     .when(pl.col("desc_age_month_lo").is_not_null() & pl.col("desc_age_month_hi").is_null()).then(pl.lit(0.85))
     .otherwise(pl.lit(None))
     .alias("desc_age_confidence")
)

# 5) điền bổ sung vào tuổi chính (coalesce ưu tiên primary)
df = df.with_columns([
    pl.when(pl.col("age_bucket_primary_filled").is_not_null())
      .then(pl.col("age_bucket_primary_filled"))
      .when(pl.col("desc_age_bucket").is_not_null())
      .then(pl.col("desc_age_bucket"))
      .otherwise(pl.lit(None))
      .alias("age_bucket_enriched"),
    pl.when(pl.col("age_bucket_primary_filled").is_not_null()).then(pl.lit("primary"))
     .when(pl.col("desc_age_bucket").is_not_null()).then(pl.lit("from_description"))
     .otherwise(pl.lit("missing"))
     .alias("age_source_enriched")
])

# 6) báo cáo
report = (
    df.select([
        pl.count().alias("total"),
        pl.col("age_bucket_primary_filled").is_not_null().sum().alias("present_primary"),
        pl.col("age_bucket_enriched").is_not_null().sum().alias("present_enriched"),
        (pl.col("age_source_enriched") == "from_description").sum().alias("filled_from_description"),
    ])
    .with_columns(
        primary_pct   = (pl.col("present_primary")   / pl.col("total") * 100).round(2),
        enriched_pct  = (pl.col("present_enriched")  / pl.col("total") * 100).round(2),
        added_by_desc = (pl.col("filled_from_description") / pl.col("total") * 100).round(2),
    )
)
print(">>> Age coverage (before/after + added by description)")
print(report)

# vài dòng mẫu điền từ mô tả
print("\n>>> Sample rows filled from description:")
print(
    df.filter(pl.col("age_source_enriched") == "from_description")
      .select(["desc_age_bucket","desc_age_confidence","description_merge"])
      .head(5)
)

# lưu lại
df_cleaned = df
print("\nĐã cập nhật df_cleaned với cột 'age_bucket_enriched' và 'age_source_enriched'")

>>> Age coverage (before/after + added by description)
shape: (1, 7)
┌───────┬─────────────────┬──────────────────┬─────────────────────────┬─────────────┬──────────────┬───────────────┐
│ total ┆ present_primary ┆ present_enriched ┆ filled_from_description ┆ primary_pct ┆ enriched_pct ┆ added_by_desc │
│ ---   ┆ ---             ┆ ---              ┆ ---                     ┆ ---         ┆ ---          ┆ ---           │
│ u32   ┆ u32             ┆ u32              ┆ u32                     ┆ f64         ┆ f64          ┆ f64           │
╞═══════╪═════════════════╪══════════════════╪═════════════════════════╪═════════════╪══════════════╪═══════════════╡
│ 27332 ┆ 12359           ┆ 17150            ┆ 4791                    ┆ 45.22       ┆ 62.75        ┆ 17.53         │
└───────┴─────────────────┴──────────────────┴─────────────────────────┴─────────────┴──────────────┴───────────────┘

>>> Sample rows filled from description:
shape: (5, 3)
┌─────────────────┬─────────────────────┬────────

C:\Users\PC\AppData\Local\Temp\ipykernel_10564\330814489.py:94: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  pl.count().alias("total"),


In [160]:

pl.Config.set_tbl_rows(50)             # số dòng tối đa hiển thị
pl.Config.set_tbl_cols(30)             # số cột tối đa hiển thị
pl.Config.set_tbl_width_chars(240)     # tổng chiều rộng bảng ký tự
pl.Config.set_fmt_str_lengths(1000)    # <== độ dài chuỗi tối đa hiển thị (quan trọng nhất!)

# --- Lọc 10 dòng được fill từ description_merge ---
filled_from_desc = (
    df_cleaned
    .filter(pl.col("age_source_enriched") == "from_description")
    .select([
        "age_bucket_enriched",
        "desc_age_confidence",
        "description_merge"
    ])
    .head(10)
)

# --- In ra ---
print(">>> 10 dòng được fill từ description_merge (kiểm tra thủ công):")
print(filled_from_desc)


>>> 10 dòng được fill từ description_merge (kiểm tra thủ công):
shape: (10, 3)
┌─────────────────────┬─────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│ age_bucket_enriched ┆ desc_age_confidence ┆ description_merge                                                                                                                                                                                │
│ ---                 ┆ ---                 ┆ ---                                                                                                                                                                                              │
│ str                 ┆ f64                 ┆ str                                                                                                                                                     

Trích xuất từ cân nặng để fill dữ liệu cho age_bucket

In [161]:
# === Trích cân nặng từ description_merge và điền bổ sung age_bucket ===

df = df_cleaned  # dùng biến chính của bạn

# 0) Chuẩn hoá mô tả (nếu chưa có _desc_norm từ trước)
if "_desc_norm" not in df.columns:
    df = df.with_columns(
        pl.col("description_merge").cast(pl.Utf8)
          .str.to_lowercase()
          .str.replace_all('"', '').str.replace_all('“|”', '')
          .str.replace_all(r"\s+", " ").str.strip_chars()
          .alias("_desc_norm")
    )

# 1) Trích cân nặng (kg)
#   - Hyphen/ndashes: "4-8kg" hoặc "4 - 8 kg"
kg_r_lo = pl.col("_desc_norm").str.extract(r'(\d+(?:[.,]\d+)?)\s*[-–—]\s*(\d+(?:[.,]\d+)?)\s*kg\b', 1)
kg_r_hi = pl.col("_desc_norm").str.extract(r'(\d+(?:[.,]\d+)?)\s*[-–—]\s*(\d+(?:[.,]\d+)?)\s*kg\b', 2)

#   - "từ ... đến ... kg"
kg_tu_lo = pl.col("_desc_norm").str.extract(r'(?:từ)\s*(\d+(?:[.,]\d+)?)\s*(?:kg)?\s*(?:đến|to|-|–|—)\s*(\d+(?:[.,]\d+)?)\s*kg\b', 1)
kg_tu_hi = pl.col("_desc_norm").str.extract(r'(?:từ)\s*(\d+(?:[.,]\d+)?)\s*(?:kg)?\s*(?:đến|to|-|–|—)\s*(\d+(?:[.,]\d+)?)\s*kg\b', 2)

#   - Cân nặng đơn: "8 kg"
kg_single = pl.col("_desc_norm").str.extract(r'\b(\d+(?:[.,]\d+)?)\s*kg\b', 1)

def to_float(expr: pl.Expr) -> pl.Expr:
    return expr.str.replace_all(",", ".").cast(pl.Float64)

df = df.with_columns([
    pl.coalesce([to_float(kg_r_lo), to_float(kg_tu_lo)]).alias("kg_lo"),
    pl.coalesce([to_float(kg_r_hi), to_float(kg_tu_hi)]).alias("kg_hi"),
    to_float(kg_single).alias("kg_single"),
])

# Nếu có khoảng thì dùng khoảng; nếu không thì dùng đơn lẻ
df = df.with_columns([
    pl.when(pl.col("kg_lo").is_not_null() & pl.col("kg_hi").is_not_null())
      .then((pl.col("kg_lo") + pl.col("kg_hi")) / 2)
      .when(pl.col("kg_lo").is_not_null() & pl.col("kg_hi").is_null())
      .then(pl.col("kg_lo"))
      .when(pl.col("kg_lo").is_null() & pl.col("kg_hi").is_null() & pl.col("kg_single").is_not_null())
      .then(pl.col("kg_single"))
      .otherwise(None)
      .alias("kg_mid")
])

# 2) Map kg_mid -> age bucket theo heuristic
def bucket_from_kg_expr(kg_mid: pl.Expr) -> pl.Expr:
    return (
        pl.when(kg_mid.is_null()).then(pl.lit(None))
         .when(kg_mid <= 6).then(pl.lit("0-6M"))
         .when(kg_mid < 7.5).then(pl.lit("3-6M"))   # nội bộ; sẽ map sang 5 bucket chính
         .when(kg_mid < 9.5).then(pl.lit("6-9M"))
         .when(kg_mid < 11.5).then(pl.lit("9-12M"))
         .when(kg_mid < 14).then(pl.lit("1-2Y"))
         .when(kg_mid < 20).then(pl.lit("2-4Y"))
         .otherwise(pl.lit(">4Y"))
    )

df = df.with_columns(
    bucket_from_kg_expr(pl.col("kg_mid")).alias("weight_age_bucket_raw")
)

# Gom về lưới bucket chính 5 mức
def normalize_to_primary(bucket_col: pl.Expr) -> pl.Expr:
    return (
        pl.when(bucket_col.is_null()).then(pl.lit(None))
         .when(bucket_col == "0-6M").then(pl.lit("0-6M"))
         .when((bucket_col == "3-6M") | (bucket_col == "6-9M") | (bucket_col == "9-12M"))
            .then(pl.lit("6-12M"))
         .when(bucket_col == "1-2Y").then(pl.lit("1-2Y"))
         .when(bucket_col == "2-4Y").then(pl.lit("2-4Y"))
         .otherwise(pl.lit(">4Y"))
    )

df = df.with_columns(
    normalize_to_primary(pl.col("weight_age_bucket_raw")).alias("weight_age_bucket")
)

# 3) Confidence & source (range > single)
df = df.with_columns(
    pl.when(pl.col("kg_lo").is_not_null() & pl.col("kg_hi").is_not_null()).then(pl.lit(0.70))
     .when(pl.col("kg_mid").is_not_null()).then(pl.lit(0.55))
     .otherwise(pl.lit(None)).alias("weight_age_confidence"),
).with_columns(
    pl.when(pl.col("kg_lo").is_not_null() & pl.col("kg_hi").is_not_null()).then(pl.lit("weight_range"))
     .when(pl.col("kg_mid").is_not_null()).then(pl.lit("weight_single"))
     .otherwise(pl.lit(None)).alias("weight_age_source")
)

# 4) Điền bổ sung vào tuổi (sau bước enrich từ description)
df = df.with_columns([
    pl.when(pl.col("age_bucket_enriched").is_not_null())
      .then(pl.col("age_bucket_enriched"))
      .when(pl.col("weight_age_bucket").is_not_null())
      .then(pl.col("weight_age_bucket"))
      .otherwise(pl.lit(None))
      .alias("age_bucket_final"),
    pl.when(pl.col("age_bucket_enriched").is_not_null()).then(pl.lit("enriched"))
     .when(pl.col("weight_age_bucket").is_not_null()).then(pl.lit("from_weight"))
     .otherwise(pl.lit("missing"))
     .alias("age_source_final")
])

# 5) Báo cáo uplift từ cân nặng
report_w = (
    df.select([
        pl.len().alias("total"),
        pl.col("age_bucket_enriched").is_not_null().sum().alias("present_before"),
        pl.col("age_bucket_final").is_not_null().sum().alias("present_after"),
        (pl.col("age_source_final") == "from_weight").sum().alias("filled_from_weight"),
    ])
    .with_columns(
        before_pct = (pl.col("present_before") / pl.col("total") * 100).round(2),
        after_pct  = (pl.col("present_after")  / pl.col("total") * 100).round(2),
        added_pct  = (pl.col("filled_from_weight") / pl.col("total") * 100).round(2),
    )
)

print(">>> Weight-based enrichment report")
print(report_w)

print("\n>>> Sample rows filled from weight:")
print(
    df.filter(pl.col("age_source_final") == "from_weight")
      .select(["kg_lo","kg_hi","kg_mid","weight_age_bucket","weight_age_confidence","description_merge"])
      .head(10)
)

# Cập nhật lại DataFrame chính
df_cleaned = df


>>> Weight-based enrichment report
shape: (1, 7)
┌───────┬────────────────┬───────────────┬────────────────────┬────────────┬───────────┬───────────┐
│ total ┆ present_before ┆ present_after ┆ filled_from_weight ┆ before_pct ┆ after_pct ┆ added_pct │
│ ---   ┆ ---            ┆ ---           ┆ ---                ┆ ---        ┆ ---       ┆ ---       │
│ u32   ┆ u32            ┆ u32           ┆ u32                ┆ f64        ┆ f64       ┆ f64       │
╞═══════╪════════════════╪═══════════════╪════════════════════╪════════════╪═══════════╪═══════════╡
│ 27332 ┆ 17150          ┆ 18384         ┆ 1234               ┆ 62.75      ┆ 67.26     ┆ 4.51      │
└───────┴────────────────┴───────────────┴────────────────────┴────────────┴───────────┴───────────┘

>>> Sample rows filled from weight:
shape: (10, 6)
┌───────┬───────┬────────┬───────────────────┬───────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [162]:
df_cleaned.columns

['item_id',
 'price',
 'category_l1',
 'category',
 'brand',
 'gender_target',
 'item_type',
 'color',
 'size',
 'origin',
 'material',
 'sale_status',
 'price_capped',
 'description_merge',
 'age_bucket_primary',
 'size_lo_m',
 'size_hi_m',
 'size_age_bucket',
 'age_fill_source',
 'age_bucket_primary_filled',
 '_desc_norm',
 'desc_age_month_lo',
 'desc_age_month_hi',
 'desc_age_month_mid',
 'desc_age_bucket',
 'desc_age_confidence',
 'age_bucket_enriched',
 'age_source_enriched',
 'kg_lo',
 'kg_hi',
 'kg_single',
 'kg_mid',
 'weight_age_bucket_raw',
 'weight_age_bucket',
 'weight_age_confidence',
 'weight_age_source',
 'age_bucket_final',
 'age_source_final']

Xóa các cột trung gian để cho bước xử lý tiếp theo. Cột size thiếu quá nhiều dữ liệu, cũng cần xóa luôn

In [163]:
# === Giữ lại các cột chính cần thiết ===
keep_cols = [
    "item_id",
    "price",
    "category_l1",
    "category",
    "brand",
    "gender_target",
    "item_type",
    "color",
    "origin",
    "material",
    "sale_status",
    "description_merge",
    "age_bucket_final",
]

# Giữ lại & loại bỏ phần còn lại
df_cleaned = df_cleaned.select([c for c in keep_cols if c in df_cleaned.columns])

print("Đã dọn cột trung gian, chỉ giữ lại các cột quan trọng:")
print(df_cleaned.columns)
print(f"Tổng số cột còn lại: {len(df_cleaned.columns)}")


Đã dọn cột trung gian, chỉ giữ lại các cột quan trọng:
['item_id', 'price', 'category_l1', 'category', 'brand', 'gender_target', 'item_type', 'color', 'origin', 'material', 'sale_status', 'description_merge', 'age_bucket_final']
Tổng số cột còn lại: 13


### Các feature khác ( brand, item_type, color, origin, material)

Muốn fill dữ liệu cho các cột trên, trước tiên tôi tiến hành việc chuẩn bị từ điển alias LEXICONS

In [164]:
# === Cell: Chuẩn hoá & tạo từ điển (brand, gender_target, item_type, color, origin, material) ===
import unicodedata, re, string
import polars as pl

df = df_cleaned  # DataFrame hiện tại của bạn

# -----------------------------
# 0️⃣ Helpers chuẩn hoá & missing
# -----------------------------
def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

def norm_text(s: str) -> str:
    s = str(s).lower()
    s = s.replace("–", "-").replace("—", "-").replace("−", "-")
    s = re.sub(r"[“”\"“”]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_key(s: str, compact: bool = False) -> str:
    s = norm_text(s)
    s = strip_accents(s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    s = re.sub(r"\s+", "" if compact else " ", s)
    return s

def is_missing_like(s: str | None) -> bool:
    if s is None:
        return True
    t = norm_key(s)
    return t in {"khong xac dinh", "khongxd", "unknown", "undef", "chua ro", "na", "n a", "none", ""}

def make_variants(name: str) -> set[str]:
    if not name:
        return set()
    raw = str(name)
    n = norm_text(raw)
    nd = strip_accents(n)
    v = {n, nd, norm_key(raw), norm_key(raw, compact=True)}
    v.add(nd.replace("'", "").replace("&", " ").replace(".", " ").replace("-", " "))
    return {re.sub(r"\s+", " ", x).strip() for x in v if x}

def compile_pattern(aliases: set[str]) -> re.Pattern | None:
    if not aliases:
        return None
    alts = []
    for a in sorted(aliases, key=lambda x: (-len(x), x)):
        a_nd = strip_accents(norm_text(a))
        a_nd = re.sub(r"\s+", r"\\s+", re.escape(a_nd))
        alts.append(a_nd)
    return re.compile(r"\b(?:%s)\b" % "|".join(alts), flags=re.IGNORECASE)

def compile_pattern_vi(aliases: set[str]) -> re.Pattern | None:
    """Compile pattern GIỮ DẤU (cho color)."""
    if not aliases:
        return None
    alts = []
    for a in sorted(aliases, key=lambda x: (-len(x), x)):
        a_vi = re.sub(r"\s+", r"\\s+", re.escape(a.strip()))
        alts.append(a_vi)
    return re.compile(rf"(?<![A-Za-zÀ-ỹ])(?:{'|'.join(alts)})(?![A-Za-zÀ-ỹ])", flags=re.IGNORECASE | re.UNICODE)

# -----------------------------
# 1️⃣ Chuẩn hoá mô tả để parse sau này
# -----------------------------
df = df.with_columns([
    pl.col("description_merge").cast(pl.Utf8).alias("description_merge"),
    pl.col("description_merge").map_elements(lambda x: norm_text(x) if x is not None else None, return_dtype=pl.Utf8).alias("desc_norm"),
    pl.col("description_merge").map_elements(lambda x: strip_accents(norm_text(x)) if x is not None else None, return_dtype=pl.Utf8).alias("desc_nodiac"),
])

# -----------------------------
# 2️⃣ Từ điển/lexicon cho từng field
# -----------------------------
brand_alias_map = {
    "DrBrown's": {"drbrown's", "dr brown", "drbrowns", "dr brown's", "drbrown"},
    "Merries": {"merries"},
    "Huggies": {"huggies"},
    "Moony": {"moony"},
    "Pampers": {"pampers"},
    "Goon": {"goon", "goo.n"},
    "Bobby": {"bobby"},
    "Pigeon": {"pigeon"},
    "Chicco": {"chicco"},
    "Philips Avent": {"philips avent", "avent", "philips-avent"},
    "Comotomo": {"comotomo"},
    "Babycare": {"babycare"},
    "Fisher-Price": {"fisher price", "fisherprice"},
    "Mamamy": {"mamamy"},
    "Unicharm": {"unicharm"},
    "Kao": {"kao"},
    "Johnson's": {"johnson's", "johnsons", "johnson and johnson"},
    "Meiji": {"meiji"},
}

gender_alias_map = {
    "Bé Trai": {"bé trai", "be trai", "boy", "nam", "male"},
    "Bé Gái": {"bé gái", "be gai", "girl", "nu", "nữ", "female"},
    "Unisex": {"unisex", "cho bé", "tre em", "trẻ em", "kid", "kids", "baby", "sơ sinh", "so sinh", "newborn"},
}

item_type_alias_map = {
    "Tã dán": {"tã dán", "ta dan", "bỉm dán", "bim dan", "taped diaper", "diaper tape"},
    "Tã quần": {"tã quần", "ta quan", "bỉm quần", "bim quan", "pants diaper", "pull-up"},
    "Núm ty": {"núm ty", "num ty", "núm vú", "num vu", "nipple"},
    "Bình sữa": {"bình sữa", "binh sua", "chai sữa", "feeding bottle", "bottle"},
    "Khăn ướt": {"khăn ướt", "khan uot", "wet wipes", "wipes", "khăn giấy ướt"},
    "Quần áo": {"quần áo", "quan ao", "áo", "bodysuit", "romper", "onesie"},
    "Sữa tắm": {"sữa tắm", "sua tam", "bath gel", "body wash", "shampoo"},
    "Xe đẩy": {"xe đẩy", "xe day", "stroller"},
    "Ghế ngồi ô tô": {"ghế ngồi ô tô", "ghe ngoi o to", "car seat", "ghe oto"},
}

color_alias_map = {
    "Đen": {"đen", "black", "bk"},
    "Trắng": {"trắng", "trang", "white", "wh"},
    "Xám": {"xám", "xam", "ghi", "gray", "grey"},
    "Đỏ": {"đỏ", "red"},
    "Cam": {"cam", "orange"},
    "Vàng": {"vàng", "vang", "yellow"},
    "Xanh lá": {"xanh lá", "xanh la", "green"},
    "Xanh dương": {"xanh dương", "xanh duong", "blue", "navy", "xanh bien"},
    "Nâu": {"nâu", "brown", "tan"},
    "Hồng": {"hồng", "pink", "rose"},
    "Tím": {"tím", "purple", "violet"},
    "Bạc": {"bạc", "silver", "xam bac", "silver gray"},
    "Vàng kim": {"vàng kim", "vang kim", "gold", "golden"},
}

material_alias_map = {
    "Cotton": {"cotton", "coton", "bông", "bong", "organic cotton", "cotton hữu cơ", "cotton huu co"},
    "Polyester": {"polyester", "polieste", "poly"},
    "Spandex": {"spandex", "elastane", "lycra"},
    "Nylon": {"nylon"},
    "Da": {"da", "leather"},
    "Gỗ": {"gỗ", "go", "wood"},
    "Inox/Thép không gỉ": {"inox", "stainless steel", "thép không gỉ", "thep khong gi", "thep", "steel"},
    "Silicone": {"silicone", "silicon"},
    "PP": {"pp", "polypropylene"},
    "PE": {"pe", "polyethylene"},
    "ABS": {"abs"},
    "PC": {"pc", "polycarbonate"},
    "Thủy tinh": {"thủy tinh", "thuy tinh", "kính", "kinh", "glass"},
    "Len": {"len", "wool"},
    "Cao su": {"cao su", "rubber"},
    "Nỉ": {"nỉ", "ni", "fleece"},
}

origin_alias_map = {
    "Mỹ": {"mỹ", "my", "usa", "us", "united states", "hoa ky", "hoa kỳ"},
    "Nhật Bản": {"nhật", "nhat", "nhật bản", "japan", "jp"},
    "Trung Quốc": {"trung quốc", "trung quoc", "china", "cn", "prc"},
    "Hàn Quốc": {"hàn quốc", "han quoc", "korea", "kr", "south korea"},
    "Việt Nam": {"việt nam", "viet nam", "vietnam", "vn"},
    "Thái Lan": {"Thai","thái lan", "thai lan", "thailand"},
    "Đức": {"đức", "duc", "germany", "de"},
    "Pháp": {"pháp", "phap", "france", "fr"},
    "Anh": {"anh", "uk", "united kingdom", "great britain", "gb"},
    "Thái Lan": {"thái lan", "thai lan", "thailand"},
    "Ý": {"ý", "italy"},
}

# -----------------------------
# 3️⃣ Chuẩn hoá alias & pattern
# -----------------------------
def expand_alias_map(alias_map: dict[str, set[str]], keep_diacritic: bool = False):
    canon_to_aliases, alias_to_canon, canon_to_pattern = {}, {}, {}
    for canon, aliases in alias_map.items():
        canon_disp = canon.strip()
        bag = set()
        for a in (aliases | {canon_disp}):
            bag |= make_variants(a)
        canon_to_aliases[canon_disp] = bag
        for a in bag:
            key = norm_key(a)
            alias_to_canon[key] = canon_disp
            alias_to_canon[norm_key(a, compact=True)] = canon_disp
        pat = compile_pattern_vi(bag) if keep_diacritic else compile_pattern(bag)
        if pat:
            canon_to_pattern[canon_disp] = pat
    return canon_to_aliases, alias_to_canon, canon_to_pattern

brand_canon_aliases, brand_alias_to_canon, brand_patterns = expand_alias_map(brand_alias_map)
gender_canon_aliases, gender_alias_to_canon, gender_patterns = expand_alias_map(gender_alias_map)
itemtype_canon_aliases, itemtype_alias_to_canon, itemtype_patterns = expand_alias_map(item_type_alias_map)
color_canon_aliases, color_alias_to_canon, color_patterns = expand_alias_map(color_alias_map, keep_diacritic=True)
material_canon_aliases, material_alias_to_canon, material_patterns = expand_alias_map(material_alias_map)
origin_canon_aliases, origin_alias_to_canon, origin_patterns = expand_alias_map(origin_alias_map)

# -----------------------------
# 4️⃣ Flag thiếu gốc
# -----------------------------
def mk_missing_flag(col: str) -> pl.Expr:
    return pl.col(col).is_null() | pl.col(col).map_elements(is_missing_like, return_dtype=pl.Boolean)

df = df.with_columns([
    mk_missing_flag("brand").alias("brand_missing"),
    mk_missing_flag("gender_target").alias("gender_missing"),
    mk_missing_flag("item_type").alias("item_type_missing"),
    mk_missing_flag("color").alias("color_missing"),
    mk_missing_flag("origin").alias("origin_missing"),
    mk_missing_flag("material").alias("material_missing"),
])

# -----------------------------
# 5️⃣ Tập hợp từ điển LEXICONS
# -----------------------------
LEXICONS = {
    "brand": {"canon_aliases": brand_canon_aliases, "alias_to_canon": brand_alias_to_canon, "patterns": brand_patterns},
    "gender_target": {"canon_aliases": gender_canon_aliases, "alias_to_canon": gender_alias_to_canon, "patterns": gender_patterns},
    "item_type": {"canon_aliases": itemtype_canon_aliases, "alias_to_canon": itemtype_alias_to_canon, "patterns": itemtype_patterns},
    "color": {"canon_aliases": color_canon_aliases, "alias_to_canon": color_alias_to_canon, "patterns": color_patterns},
    "material": {"canon_aliases": material_canon_aliases, "alias_to_canon": material_alias_to_canon, "patterns": material_patterns},
    "origin": {"canon_aliases": origin_canon_aliases, "alias_to_canon": origin_alias_to_canon, "patterns": origin_patterns},
}

# -----------------------------
# 6️⃣ Xuất & lưu lại
# -----------------------------
df_lexicon = df
print("✅ LEXICONS created. Color giữ dấu, các field khác không dấu.")
print({k: len(v["canon_aliases"]) for k, v in LEXICONS.items()})


✅ LEXICONS created. Color giữ dấu, các field khác không dấu.
{'brand': 18, 'gender_target': 3, 'item_type': 9, 'color': 13, 'material': 16, 'origin': 10}


Sử dụng từ điển alias và biểu thức chính quy để dò tìm các đặc trưng (brand, color, origin, …) trong mô tả sản phẩm, sau đó tự động điền vào các ô bị thiếu, đồng thời ghi chú nguồn gốc giá trị và báo cáo mức độ cải thiện độ đầy đủ của dữ liệu.

In [165]:
# === Cell: High-recall extract + fill (sử dụng desc_nodiac + LEXICONS) ===

df = df_lexicon  # DataFrame hiện tại sau bước chuẩn hoá & có desc_nodiac

# -----------------------------
# 1️⃣ Helper: xác định missing (NULL hoặc "Không xác định")
# -----------------------------
def is_missing_expr(col: str) -> pl.Expr:
    return pl.col(col).is_null() | (pl.col(col).cast(pl.Utf8).str.strip_chars() == "Không xác định")

# -----------------------------# === Cell: High-recall extract + fill (sử dụng desc_nodiac + LEXICONS) ===

df = df_lexicon  # DataFrame hiện tại sau bước chuẩn hoá & có desc_nodiac

# -----------------------------
# 1️⃣ Helper: xác định missing (NULL hoặc "Không xác định")
# -----------------------------
def is_missing_expr(col: str) -> pl.Expr:
    return pl.col(col).is_null() | (pl.col(col).cast(pl.Utf8).str.strip_chars() == "Không xác định")

# -----------------------------
# 2️⃣ Tạo extractor cho từng field từ LEXICONS
# -----------------------------
def make_extractor(field: str):
    """Sinh hàm dò theo pattern regex từ LEXICONS[field]['patterns']"""
    patterns = LEXICONS[field]["patterns"]
    items = list(patterns.items())

    def extract_one(desc_nodiac: str | None):
        if not desc_nodiac:
            return None
        best, best_pos = None, 10**9
        for canon, pat in items:
            m = pat.search(desc_nodiac)
            if m and m.start() < best_pos:
                best, best_pos = canon, m.start()
        return best

    return extract_one

# -----------------------------
# 3️⃣ Áp dụng extractor cho từng field (tạo *_from_desc)
# -----------------------------
extractors = {f: make_extractor(f) for f in LEXICONS.keys()}

# --- Sử dụng desc_norm cho color & origin (giữ dấu), còn lại bỏ dấu ---
USE_NORM = {"color", "origin"}

for f in LEXICONS.keys():
    text_col = "desc_norm" if f in USE_NORM else "desc_nodiac"
    df = df.with_columns([
        pl.col(text_col)
        .map_elements(extractors[f], return_dtype=pl.Utf8)
        .alias(f"{f}_from_desc")
    ])


# -----------------------------
# 4️⃣ Fill giá trị thiếu (NULL / "Không xác định") bằng *_from_desc
# -----------------------------
def fill_from_desc(df: pl.DataFrame, field: str) -> pl.DataFrame:
    src = f"{field}_from_desc"
    miss = is_missing_expr(field)

    return df.with_columns([
        pl.when(~miss).then(pl.col(field))
         .when(miss & pl.col(src).is_not_null()).then(pl.col(src))
         .otherwise(None)
         .alias(f"{field}_final"),

        pl.when(~miss).then(pl.lit("original"))
         .when(miss & pl.col(src).is_not_null()).then(pl.lit("from_desc_dict"))
         .otherwise(pl.lit("missing"))
         .alias(f"{field}_fill_source"),
    ])

fields = list(LEXICONS.keys())
for f in fields:
    df = fill_from_desc(df, f)

# -----------------------------
# 5️⃣ Báo cáo thống kê kết quả fill
# -----------------------------
rows = []
for f in fields:
    before = df.select((~is_missing_expr(f)).sum().alias("v")).item()
    after  = df.select((~is_missing_expr(f"{f}_final")).sum().alias("v")).item()
    filled = df.select((is_missing_expr(f) & ~is_missing_expr(f"{f}_final")).sum().alias("v")).item()
    rows.append({
        "field": f,
        "total": df.height,
        "present_before": before,
        "present_after": after,
        "filled_from_desc": filled,
        "missing_after": df.height - after,
        "gain_pct": round(100 * filled / df.height, 2)
    })

report = pl.DataFrame(rows).sort("field")
print("\n=== Báo cáo fill từ mô tả ===")
print(report)

# -----------------------------
# 6️⃣ Lưu kết quả sau fill
# -----------------------------
df_filled = df
print("\nĐã fill dữ liệu từ description_merge thành công và lưu vào df_filled.")
# 2️⃣ Tạo extractor cho từng field từ LEXICONS
# -----------------------------
def make_extractor(field: str):
    """Sinh hàm dò theo pattern regex từ LEXICONS[field]['patterns']"""
    patterns = LEXICONS[field]["patterns"]
    items = list(patterns.items())

    def extract_one(desc_nodiac: str | None):
        if not desc_nodiac:
            return None
        best, best_pos = None, 10**9
        for canon, pat in items:
            m = pat.search(desc_nodiac)
            if m and m.start() < best_pos:
                best, best_pos = canon, m.start()
        return best

    return extract_one

# -----------------------------
# 3️⃣ Áp dụng extractor cho từng field (tạo *_from_desc)
# -----------------------------
extractors = {f: make_extractor(f) for f in LEXICONS.keys()}

for f in LEXICONS.keys():
    df = df.with_columns([
        pl.col("desc_nodiac")
        .map_elements(extractors[f], return_dtype=pl.Utf8)
        .alias(f"{f}_from_desc")
    ])

# -----------------------------
# 4️⃣ Fill giá trị thiếu (NULL / "Không xác định") bằng *_from_desc
# -----------------------------
def fill_from_desc(df: pl.DataFrame, field: str) -> pl.DataFrame:
    src = f"{field}_from_desc"
    miss = is_missing_expr(field)

    return df.with_columns([
        pl.when(~miss).then(pl.col(field))
         .when(miss & pl.col(src).is_not_null()).then(pl.col(src))
         .otherwise(None)
         .alias(f"{field}_final"),

        pl.when(~miss).then(pl.lit("original"))
         .when(miss & pl.col(src).is_not_null()).then(pl.lit("from_desc_dict"))
         .otherwise(pl.lit("missing"))
         .alias(f"{field}_fill_source"),
    ])

fields = list(LEXICONS.keys())
for f in fields:
    df = fill_from_desc(df, f)

# -----------------------------
# 5️⃣ Báo cáo thống kê kết quả fill
# -----------------------------
rows = []
for f in fields:
    before = df.select((~is_missing_expr(f)).sum().alias("v")).item()
    after  = df.select((~is_missing_expr(f"{f}_final")).sum().alias("v")).item()
    filled = df.select((is_missing_expr(f) & ~is_missing_expr(f"{f}_final")).sum().alias("v")).item()
    rows.append({
        "field": f,
        "total": df.height,
        "present_before": before,
        "present_after": after,
        "filled_from_desc": filled,
        "missing_after": df.height - after,
        "gain_pct": round(100 * filled / df.height, 2)
    })

report = pl.DataFrame(rows).sort("field")
print("\n=== Báo cáo fill từ mô tả ===")
print(report)

# -----------------------------
# 6️⃣ Lưu kết quả sau fill
# -----------------------------
df_filled = df
print("\nĐã fill dữ liệu từ description_merge thành công và lưu vào df_filled.")


=== Báo cáo fill từ mô tả ===
shape: (6, 7)
┌───────────────┬───────┬────────────────┬───────────────┬──────────────────┬───────────────┬──────────┐
│ field         ┆ total ┆ present_before ┆ present_after ┆ filled_from_desc ┆ missing_after ┆ gain_pct │
│ ---           ┆ ---   ┆ ---            ┆ ---           ┆ ---              ┆ ---           ┆ ---      │
│ str           ┆ i64   ┆ i64            ┆ i64           ┆ i64              ┆ i64           ┆ f64      │
╞═══════════════╪═══════╪════════════════╪═══════════════╪══════════════════╪═══════════════╪══════════╡
│ brand         ┆ 27332 ┆ 21852          ┆ 21857         ┆ 5                ┆ 5475          ┆ 0.02     │
│ color         ┆ 27332 ┆ 724            ┆ 7280          ┆ 6556             ┆ 20052         ┆ 23.99    │
│ gender_target ┆ 27332 ┆ 9294           ┆ 14065         ┆ 4771             ┆ 13267         ┆ 17.46    │
│ item_type     ┆ 27332 ┆ 17512          ┆ 17925         ┆ 413              ┆ 9407          ┆ 1.51     │
│ material

In [166]:
# === Cell: Giữ lại các cột chính, loại bỏ cột trung gian ===
df = df_filled  # dataframe hiện tại sau khi fill xong

# Danh sách các cột cuối cùng cần giữ
final_columns = [
    "item_id",
    "price",
    "category_l1",
    "category",
    "brand_final",
    "gender_target_final",
    "item_type_final",
    "color_final",
    "origin_final",
    "material_final",
    "sale_status",
    "description_merge",
    "age_bucket_final",
]

# Chỉ giữ các cột này nếu tồn tại (tránh lỗi nếu thiếu)
final_columns = [c for c in final_columns if c in df.columns]

# Lọc dataframe
df_final = df.select(final_columns)

# In thông tin kiểm tra
print("✅ Đã loại bỏ các cột trung gian.")
print(f"Số cột còn lại: {len(df_final.columns)}")
print("Các cột giữ lại:", df_final.columns)


✅ Đã loại bỏ các cột trung gian.
Số cột còn lại: 13
Các cột giữ lại: ['item_id', 'price', 'category_l1', 'category', 'brand_final', 'gender_target_final', 'item_type_final', 'color_final', 'origin_final', 'material_final', 'sale_status', 'description_merge', 'age_bucket_final']


## Xử lý null đối với cột gender_target

In [173]:
# === Cell: Fill target_user_group_final (rule-based from desc_nodiac & related fields) ===

print("🔧 Bắt đầu fill dữ liệu cho target_user_group_final...")

# --------------------------------------------------
# 1️⃣ Helper: xác định missing (NULL hoặc 'Không xác định')
# --------------------------------------------------
def is_missing_expr(col: str) -> pl.Expr:
    return (
        pl.col(col).is_null()
        | (pl.col(col).cast(pl.Utf8).str.strip_chars() == "Không xác định")
    )


# --------------------------------------------------
# 2️⃣ Helper: hàm infer theo rule logic mở rộng
# --------------------------------------------------
import re

def infer_target_user(
    desc_nodiac: str | None,
    category: str | None,
    item_type: str | None,
    age_bucket: str | None,
    category_l1: str | None,
) -> str | None:
    """Suy luận đối tượng người dùng mục tiêu (Target user group)"""
    desc = (desc_nodiac or "").lower()
    cat = (category or "").lower()
    item = (item_type or "").lower()
    combined = " ".join([desc, cat, item])
    l1 = (category_l1 or "").lower()
    age = (age_bucket or "").lower()

    # --- Keyword mô tả ---
    if re.search(r"(sơ\s*sinh|newborn|bé\s*sơ\s*sinh|0[-–_]6m|0[-–_]?12m)", combined):
        return "Sơ sinh"
    if re.search(r"(bé\s*gái|for\s*girl|girl|áo\s*bé\s*gái|đầm)", combined):
        return "Bé Gái"
    if re.search(r"(bé\s*trai|for\s*boy|boy|áo\s*sơ\s*mi\s*bé\s*trai|quần\s*bé\s*trai)", combined):
        return "Bé Trai"
    if re.search(r"(mẹ|cho\s*mẹ|bầu|mang\s*thai|sau\s*sinh|mẹ\s*bầu)", combined):
        return "Phụ nữ trưởng thành"
    if re.search(r"(phụ\s*nữ|lady|women|woman|nữ\s*giới|dành\s*cho\s*nữ)", combined):
        return "Phụ nữ trưởng thành"
    if re.search(r"(nam|men|male|for\s*men|đàn\s*ông|ông)", combined):
        return "Đàn ông trưởng thành"
    if re.search(r"(unisex|cả\s*nhà|gia\s*đình|family|dùng\s*chung)", combined):
        return "Unisex"

    # --- Fallback theo category_l1 ---
    if l1 in ["babycare", "tã", "hóa mỹ phẩm cho bé", "thực phẩm cho bé", "sữa"]:
        return "Sơ sinh"
    if l1 in ["thực phẩm cho gia đình", "hóa mỹ phẩm gia đình", "vệ sinh", "tpcn", "phụ kiện", "textile"]:
        return "Unisex"
    if l1 == "thời trang":
        return "Unisex"

    # --- Fallback theo age bucket ---
    if age in ["0-6m", "6-12m"]:
        return "Sơ sinh"
    if age in ["mẹ"]:
        return "Phụ nữ trưởng thành"
    if age in ["người lớn"]:
        return "Unisex"

    return None


# --------------------------------------------------
# 3️⃣ Áp dụng rule-based extractor
# --------------------------------------------------
df = df.with_columns([
    pl.struct([
        pl.col("desc_nodiac"),
        pl.col("category"),
        pl.col("item_type_final"),
        pl.col("age_bucket_final"),
        pl.col("category_l1"),
    ])
    .map_elements(
        lambda x: infer_target_user(
            x["desc_nodiac"], x["category"], x["item_type_final"], x["age_bucket_final"], x["category_l1"]
        ),
        return_dtype=pl.Utf8,
    )
    .alias("target_user_from_rule")
])


# --------------------------------------------------
# 4️⃣ Fill giá trị thiếu bằng rule result
# --------------------------------------------------
miss = is_missing_expr("gender_target_final")

df = df.with_columns([
    pl.when(~miss).then(pl.col("gender_target_final"))
     .when(miss & pl.col("target_user_from_rule").is_not_null())
     .then(pl.col("target_user_from_rule"))
     .otherwise(None)
     .alias("target_user_group_final"),

    pl.when(~miss).then(pl.lit("original"))
     .when(miss & pl.col("target_user_from_rule").is_not_null())
     .then(pl.lit("from_desc_rule"))
     .otherwise(pl.lit("missing"))
     .alias("target_user_group_fill_source"),
])


# --------------------------------------------------
# 5️⃣ Báo cáo thống kê fill
# --------------------------------------------------
before = df.select((~is_missing_expr("gender_target_final")).sum().alias("present_before")).item()
after = df.select((~is_missing_expr("target_user_group_final")).sum().alias("present_after")).item()
filled = df.select((is_missing_expr("gender_target_final") & ~is_missing_expr("target_user_group_final")).sum().alias("filled")).item()

print("\n=== Báo cáo fill target_user_group_final ===")
print(f"Tổng dòng: {df.height}")
print(f"Trước fill  : {before}")
print(f"Sau fill    : {after}")
print(f"Đã fill thêm: {filled} ({round(100 * filled / df.height, 2)}%)")

print("\n✅ Đã fill dữ liệu cho target_user_group_final thành công và lưu vào target_user_group_final.")


🔧 Bắt đầu fill dữ liệu cho target_user_group_final...

=== Báo cáo fill target_user_group_final ===
Tổng dòng: 27332
Trước fill  : 14065
Sau fill    : 25729
Đã fill thêm: 11664 (42.68%)

✅ Đã fill dữ liệu cho target_user_group_final thành công và lưu vào target_user_group_final.


In [174]:
# === Cell: Kiểm tra kết quả fill target_user_group_final (phiên bản gọn, không icon) ===

print("=== Kiểm tra các dòng đã được fill cho target_user_group_final ===")

# Lọc ra các dòng đã được fill từ rule
df_check = df.filter(pl.col("target_user_group_fill_source") == "from_desc_rule")

# Lấy 10 dòng đầu tiên và in ra thông tin quan trọng
rows = df_check.select([
    "item_id",
    "category_l1",
    "category",
    "item_type_final",
    "target_user_group_final",
    "target_user_group_fill_source",
    "desc_nodiac"
]).head(10).to_dicts()

for i, row in enumerate(rows, 1):
    print(f"\n--- Row {i} ---")
    print(f"item_id: {row.get('item_id', '')}")
    print(f"category_l1: {row.get('category_l1', '')}")
    print(f"category: {row.get('category', '')}")
    print(f"item_type_final: {row.get('item_type_final', '')}")
    print(f"target_user_group_final: {row.get('target_user_group_final', '')}")
    print(f"fill_source: {row.get('target_user_group_fill_source', '')}")

    desc = row.get('desc_nodiac')
    if desc:
        desc_short = desc[:180].replace("\n", " ")
    else:
        desc_short = "(no description)"
    print(f"description: {desc_short}...")


=== Kiểm tra các dòng đã được fill cho target_user_group_final ===

--- Row 1 ---
item_id: 0502020000004
category_l1: Babycare
category: Núm ty Dr Brown
item_type_final: None
target_user_group_final: Sơ sinh
fill_source: from_desc_rule
description: chi tiet san pham ten san pham: hop 2 num ty drbrown's options plus co rong y cut (tren 9 thang) loai san pham: num ty đoi tuong su dung: tre tren 9 thang thuong hieu: drbrown's đi...

--- Row 2 ---
item_id: 0020010000094
category_l1: Tã
category: Merries_Sơ Sinh
item_type_final: None
target_user_group_final: Sơ sinh
fill_source: from_desc_rule
description: ﻿﻿ta dan merries size s 82 mieng la san pham danh cho be 4-8kg đen tu thuong hieu uy tin merries cua nhat ban. ra đoi voi cong nghe 4 sieu - sieu mem mai, sieu thong thoang, sieu t...

--- Row 3 ---
item_id: 0020010000098
category_l1: Tã
category: Merries_Tã Quần
item_type_final: None
target_user_group_final: Sơ sinh
fill_source: from_desc_rule
description: ﻿﻿﻿bim ta quan merries size m 5

In [175]:
freq_table = (
    df.select(pl.col("target_user_group_final").value_counts())
      .unnest("target_user_group_final")
      .sort("count", descending=True)
      .head(10)
)

print(freq_table)

shape: (7, 2)
┌─────────────────────────┬───────┐
│ target_user_group_final ┆ count │
│ ---                     ┆ ---   │
│ str                     ┆ u32   │
╞═════════════════════════╪═══════╡
│ Bé Trai                 ┆ 8679  │
│ Bé Gái                  ┆ 5899  │
│ Sơ sinh                 ┆ 5843  │
│ Unisex                  ┆ 4485  │
│ null                    ┆ 1603  │
│ Đàn ông trưởng thành    ┆ 646   │
│ Phụ nữ trưởng thành     ┆ 177   │
└─────────────────────────┴───────┘


In [178]:
# === Cell: Dọn sạch cột trung gian sau khi fill xong tất cả ===

# Danh sách cột chính thức cần giữ lại
final_columns = [
    "item_id",
    "price",
    "category_l1",
    "category",
    "brand_final",
    "target_user_group_final",
    "item_type_final",
    "color_final",
    "origin_final",
    "material_final",
    "sale_status",
    "description_merge",
    "age_bucket_final",
]

# Giữ lại các cột chính nếu có trong df
final_columns = [c for c in final_columns if c in df.columns]

# Lọc dataframe chỉ giữ các cột cần thiết
df_final = df.select(final_columns)

print("Đã loại bỏ tất cả các cột trung gian.")
print(f"Số cột còn lại: {len(df_final.columns)}")
print("Các cột cuối cùng:", df_final.columns)


Đã loại bỏ tất cả các cột trung gian.
Số cột còn lại: 13
Các cột cuối cùng: ['item_id', 'price', 'category_l1', 'category', 'brand_final', 'target_user_group_final', 'item_type_final', 'color_final', 'origin_final', 'material_final', 'sale_status', 'description_merge', 'age_bucket_final']


### **Tiếp tục loại bỏ những cột không cần thiết**

In [179]:
# === Cell: Kiểm tra tỷ lệ NULL hoặc "Không xác định" trên 13 cột chính ===

# Danh sách các cột cần kiểm tra (đã cập nhật cột target_user_group_final)
cols_to_check = [
    "item_id",
    "price",
    "category_l1",
    "category",
    "brand_final",
    "target_user_group_final",
    "item_type_final",
    "color_final",
    "origin_final",
    "material_final",
    "sale_status",
    "description_merge",
    "age_bucket_final",
]

def null_or_unknown_rate(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    """Tính số lượng và tỷ lệ giá trị NULL hoặc 'Không xác định' cho danh sách cột."""
    total = df.height
    rows = []

    for c in cols:
        n_null = df.select(pl.col(c).is_null().sum().alias("n_null")).item()

        n_unknown = df.select(
            pl.col(c)
            .cast(pl.Utf8)
            .str.strip_chars()
            .str.to_lowercase()
            .eq("không xác định")
            .sum()
            .alias("n_unknown")
        ).item()

        n_missing = n_null + n_unknown
        pct_missing = round(100 * n_missing / total, 2)

        rows.append({
            "column": c,
            "n_null": n_null,
            "n_unknown": n_unknown,
            "missing_total": n_missing,
            "missing_pct": pct_missing,
        })

    return pl.DataFrame(rows).sort("missing_pct", descending=True)

# Gọi hàm để kiểm tra
report_missing = null_or_unknown_rate(df_final, cols_to_check)

print("=== Báo cáo tỷ lệ NULL hoặc 'Không xác định' ===")
print(report_missing)


=== Báo cáo tỷ lệ NULL hoặc 'Không xác định' ===
shape: (13, 5)
┌─────────────────────────┬────────┬───────────┬───────────────┬─────────────┐
│ column                  ┆ n_null ┆ n_unknown ┆ missing_total ┆ missing_pct │
│ ---                     ┆ ---    ┆ ---       ┆ ---           ┆ ---         │
│ str                     ┆ i64    ┆ i64       ┆ i64           ┆ f64         │
╞═════════════════════════╪════════╪═══════════╪═══════════════╪═════════════╡
│ material_final          ┆ 17063  ┆ 0         ┆ 17063         ┆ 62.43       │
│ description_merge       ┆ 15435  ┆ 0         ┆ 15435         ┆ 56.47       │
│ origin_final            ┆ 15429  ┆ 0         ┆ 15429         ┆ 56.45       │
│ color_final             ┆ 15314  ┆ 0         ┆ 15314         ┆ 56.03       │
│ item_type_final         ┆ 9407   ┆ 0         ┆ 9407          ┆ 34.42       │
│ age_bucket_final        ┆ 8948   ┆ 0         ┆ 8948          ┆ 32.74       │
│ brand_final             ┆ 5475   ┆ 0         ┆ 5475          ┆ 20

Do các cột: material_final, description_merge, origin_final, color_final có tỷ lệ null trên 50%, tôi sẽ tiến hành drop các cột này

In [182]:
# === Cell: Loại bỏ các cột có tỷ lệ null > 50% ===

cols_to_drop = [
    "material_final",
    "description_merge",
    "origin_final",
    "color_final",
]

# Loại bỏ các cột có trong dataframe
df_final = df_final.drop([c for c in cols_to_drop if c in df_final.columns])

print("Đã loại bỏ các cột có tỷ lệ missing > 50%.")
print(f"Số cột còn lại: {len(df_final.columns)}")
print("Các cột còn lại:", df_final.columns)


Đã loại bỏ các cột có tỷ lệ missing > 50%.
Số cột còn lại: 9
Các cột còn lại: ['item_id', 'price', 'category_l1', 'category', 'brand_final', 'target_user_group_final', 'item_type_final', 'sale_status', 'age_bucket_final']


### === Cell: Xử lý phần còn thiếu sau khi loại bỏ các cột >50% missing ===

In [184]:
# === Cell: Xử lý phần còn thiếu sau khi loại bỏ các cột >50% missing ===

# Fill target_user_group_final còn thiếu bằng "Unisex"
df_final = df_final.with_columns([
    pl.when(pl.col("target_user_group_final").is_null())
      .then(pl.lit("Unisex"))
      .otherwise(pl.col("target_user_group_final"))
      .alias("target_user_group_final")
])

# Fill age_bucket_final từ target_user_group_final nếu có logic rõ ràng
df_final = df_final.with_columns([
    pl.when(pl.col("age_bucket_final").is_null() & (pl.col("target_user_group_final") == "Sơ sinh"))
      .then(pl.lit("0-6M"))
     .when(pl.col("age_bucket_final").is_null() & pl.col("target_user_group_final").is_in(["Bé Trai", "Bé Gái"]))
      .then(pl.lit("1-4Y"))
     .when(pl.col("age_bucket_final").is_null() & (pl.col("target_user_group_final") == "Phụ nữ trưởng thành"))
      .then(pl.lit("Mẹ"))
     .otherwise(pl.col("age_bucket_final"))
     .alias("age_bucket_final")
])

# Không fill brand_final & item_type_final (để mô hình xử lý sau)
print("Đã xử lý missing còn lại (fill logic cơ bản cho target_user_group_final và age_bucket_final).")


Đã xử lý missing còn lại (fill logic cơ bản cho target_user_group_final và age_bucket_final).


In [186]:
# === Cell: Kiểm tra lại tỷ lệ NULL hoặc 'Không xác định' sau khi fill ===

# Hàm kiểm tra (giữ nguyên như trước)
def null_or_unknown_rate(df: pl.DataFrame, cols: list[str]) -> pl.DataFrame:
    total = df.height
    rows = []
    for c in cols:
        n_null = df.select(pl.col(c).is_null().sum().alias("n_null")).item()
        n_unknown = df.select(
            pl.col(c)
            .cast(pl.Utf8)
            .str.strip_chars()
            .str.to_lowercase()
            .eq("không xác định")
            .sum()
            .alias("n_unknown")
        ).item()
        n_missing = n_null + n_unknown
        pct_missing = round(100 * n_missing / total, 2)
        rows.append({
            "column": c,
            "n_null": n_null,
            "n_unknown": n_unknown,
            "missing_total": n_missing,
            "missing_pct": pct_missing,
        })
    return pl.DataFrame(rows).sort("missing_pct", descending=True)

# Danh sách các cột còn lại sau khi drop các cột >50% missing
cols_to_check = [
    "item_id",
    "price",
    "category_l1",
    "category",
    "brand_final",
    "target_user_group_final",
    "item_type_final",
    "sale_status",
    "age_bucket_final",
]

# Gọi hàm để kiểm tra
report_missing_after_fill = null_or_unknown_rate(df_final, cols_to_check)

print("=== Báo cáo tỷ lệ NULL hoặc 'Không xác định' SAU KHI FILL ===")
print(report_missing_after_fill)


=== Báo cáo tỷ lệ NULL hoặc 'Không xác định' SAU KHI FILL ===
shape: (9, 5)
┌─────────────────────────┬────────┬───────────┬───────────────┬─────────────┐
│ column                  ┆ n_null ┆ n_unknown ┆ missing_total ┆ missing_pct │
│ ---                     ┆ ---    ┆ ---       ┆ ---           ┆ ---         │
│ str                     ┆ i64    ┆ i64       ┆ i64           ┆ f64         │
╞═════════════════════════╪════════╪═══════════╪═══════════════╪═════════════╡
│ item_type_final         ┆ 9407   ┆ 0         ┆ 9407          ┆ 34.42       │
│ brand_final             ┆ 5475   ┆ 0         ┆ 5475          ┆ 20.03       │
│ age_bucket_final        ┆ 3472   ┆ 0         ┆ 3472          ┆ 12.7        │
│ item_id                 ┆ 0      ┆ 0         ┆ 0             ┆ 0.0         │
│ price                   ┆ 0      ┆ 0         ┆ 0             ┆ 0.0         │
│ category_l1             ┆ 0      ┆ 0         ┆ 0             ┆ 0.0         │
│ category                ┆ 0      ┆ 0         ┆ 0     

# Task 3: Phân tích tương đồng và xác định xem các thuộc tính tương tự nhau. Từ đó loại bỏ đặc trưng thừa.

Trong quá trình **EDA (Exploratory Data Analysis)**, nhóm đã thực hiện phân tích tương quan dữ liệu bằng **Cramér’s V** và **heatmap** để xác định mối quan hệ giữa các đặc trưng.  
Do đó, ở **Task 2**, nhóm gần như đã thực hiện **song song** việc **phân tích tương đồng** và **xử lý dữ liệu thiếu**, cụ thể là **fill dữ liệu từ cột `description_merge`** cho các cột như:
`brand`, `item_type`, `color`, `origin`, `material`.


#### Tóm tắt lại các nội dung liên quan đến Task 3 đã được thực hiện:

- Qua quan sát dữ liệu, nhận thấy cột **`description_merge`** chứa nhiều thông tin trùng lặp với các đặc trưng khác  
  (ví dụ: tên thương hiệu, loại sản phẩm, màu sắc, chất liệu,…).

- Vì vậy, nhóm đã **xây dựng tập từ điển (lexicon)** và **thuật toán nhận dạng (regex pattern)** để **tự động phát hiện và điền giá trị còn thiếu** cho các thuộc tính nói trên dựa vào nội dung mô tả.

- Sau khi điền đầy đủ, các **cột trung gian** (như `description`, `description_new`, `size`, …) được **xóa bỏ** để tránh dư thừa.

- Các **cột hợp nhất kết quả (`*_final`)** được **giữ lại** làm **đầu vào chính cho các bước tiền xử lý và huấn luyện mô hình** sau này.

---

> Kết quả: Bộ dữ liệu đã được tinh gọn, loại bỏ đặc trưng thừa và hợp nhất thông tin từ mô tả sản phẩm,  
> giúp giảm nhiễu và tăng tính nhất quán cho hệ thống gợi ý trong các bước tiếp theo.

# Task 4: Chuẩn hóa dữ liệu (nếu có), biến đổi dữ liệu

### Chuẩn hóa cột numeric (price)

In [188]:
# === Cell: Chuẩn hóa dữ liệu numeric (price) ===

# Loại bỏ outlier nếu cần (đã xử lý trước đó)
price_mean = df_final.select(pl.col("price").mean()).item()
price_std = df_final.select(pl.col("price").std()).item()

# Z-score normalization
df_final = df_final.with_columns([
    ((pl.col("price") - price_mean) / price_std).alias("price_norm")
])

print("Đã chuẩn hóa cột 'price' bằng Z-score normalization.")


Đã chuẩn hóa cột 'price' bằng Z-score normalization.


### Chuẩn hóa text (đưa về lowercase, loại bỏ khoảng trắng thừa)

In [190]:
# === Cell: Chuẩn hóa text cho các cột categorical ===
text_cols = [
    "category_l1", "category", "brand_final",
    "item_type_final", "target_user_group_final", "age_bucket_final"
]

for col in text_cols:
    df_final = df_final.with_columns([
        pl.col(col)
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
        .alias(col)
    ])

print("Đã chuẩn hóa định dạng text cho các cột categorical (lowercase, trim).")


Đã chuẩn hóa định dạng text cho các cột categorical (lowercase, trim).


In [191]:

# Đường dẫn thư mục & tên file
save_dir = r"D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\preprocessed-dataset"
os.makedirs(save_dir, exist_ok=True)  # tạo thư mục nếu chưa tồn tại

save_path = os.path.join(save_dir, "sales_pers.item_chunk_0.parquet")

# Lưu file parquet
df_final.write_parquet(save_path)

print(f"Đã lưu dataframe xử lý tạm thời tại:\n{save_path}")
print(f"Số dòng: {df_final.height:,} | Số cột: {df_final.width}")


Đã lưu dataframe xử lý tạm thời tại:
D:\003. HK1 - Năm 3\02. CS116 - Python cho Máy học\CS116-DoAn\preprocessed-dataset\sales_pers.item_chunk_0.parquet
Số dòng: 27,332 | Số cột: 10


# Task 5: Nhóm hãy suy nghĩ xem, với bài toán dự đoán mua hàng, ta có thể tạo mới những đặc trưng nào. Sau đó tiến hành rút trích thêm các đặc trưng. Task này rất quan trọng vì ảnh hưởng hiệu quả của hệ thống.



Hiện tại, nhóm chưa tiến hành tạo thêm đặc trưng mới nào cho item_chunk, với các lý do sau:
- Các cột còn lại sau khi xử lý (price, category_l1, category, brand_final, item_type_final, target_user_group_final, age_bucket_final, sale_status) đã phản ánh hầu hết các khía cạnh quan trọng của sản phẩm, bao gồm:
    + Phân khúc sản phẩm (qua category_l1, category)
    + Thương hiệu (brand_final)
    + Loại hàng hóa (item_type_final)
    + Đối tượng người dùng (target_user_group_final, age_bucket_final)
    + Giá và trạng thái kinh doanh (price, sale_status)
- Nhóm đã cân nhắc việc tạo thêm một số đặc trưng mở rộng (như nhóm giá, mức độ phổ biến của thương hiệu, hoặc đặc trưng tương tác giữa brand và category). Tuy nhiên, nhóm nhận thấy rằng việc này sẽ hiệu quả và khách quan hơn nếu được thực hiện sau khi merge các bảng dữ liệu (item_chunk, user_chunk, purchase_chunk) — khi đó, ta mới có thể dựa trên hành vi người dùng và tương quan mua hàng để trích xuất các đặc trưng ý nghĩa hơn.
→ Vì vậy, ở giai đoạn hiện tại, nhóm chưa tạo mới đặc trưng nào cho item_chunk,
mà chỉ dừng lại ở việc làm sạch, chuẩn hóa và giữ nguyên các đặc trưng mô tả sản phẩm cơ bản.